In [ ]:
import json
import re
import string
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from copy import deepcopy
from typing import List, Dict, Tuple
import argparse

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)
from torch.utils.data import Dataset, DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
import transformers
transformers.logging.set_verbosity_error()

torch.manual_seed(100)
np.random.seed(100)

## REWARDS

In [ ]:
import ast


def compute_reward_v8(
    triplets: List[Tuple[str, str, str]],
    q_entities: List[str],
    a_entities: List[str],
    connectivity_mode: str = "linear",
    alpha: float = 0.8,
    lambda_lin: float = 0.2,
    max_hops: int = 5,
) -> float:
    
    if not triplets:
        return 0.0

    # Build bidirectional graph
    G = nx.DiGraph()
    for s,p,o in triplets:
        s_l, o_l, p_l = s.lower(), o.lower(), p.lower()
        G.add_edge(s_l, o_l, relation=p_l)

    # 1. Fractional Answer‐Presence
    present = sum(1 for a in a_entities if a.lower() in G)
    frac_presence = present / len(a_entities)

    # 2. Graded Connectivity & 3. Efficiency (we’ll compute both from shortest paths)
    conn_score = 0.0
    #eff_score  = 0.0
    for q in q_entities:
        for a in a_entities:
            qn, an = q.lower(), a.lower()
            try:
                d = nx.shortest_path_length(G, qn, an)
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue
            conn = max(0.0, 1.0 - lambda_lin * (d-1))
            conn_score = max(conn_score, conn)

            #efficiency (inverse‐length as before)
            #eff_score = max(eff_score, 1.0 / (1 + d))

    triplet_pairs = { tuple((s.lower(), o.lower())) for s,_,o in triplets}

    cov_scores = []
    for q in q_entities:
        for a in a_entities:
            qn, an = q.lower(), a.lower()
            try:
                path = nx.shortest_path(G, qn, an)
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue
            if len(path) < 2:
                continue
            matches = sum(1 for u,v in zip(path, path[1:]) if tuple((u, v)) in triplet_pairs)
            cov_scores.append(matches / (len(path)-1))

    path_cov = max(cov_scores) if cov_scores else 0.0

    w_pres, w_conn, w_cov = 3,4,3
    total = (
        w_pres * frac_presence
      + w_conn * conn_score
      #+ w_eff  * eff_score
      + w_cov  * path_cov
    )

    return min(total, 10.0)

## Datasets

In [ ]:
class JointTrainingDatasetv3PPR(Dataset):
    def __init__(self, train_data, device='cpu'):
        self.device = device
        # This dataset is prepared using  JointTrainingDatasetv3 dataset. Only graph features are computed in this dataset
        global count_dirty_data
        self.precomputed_data = []
        for entry in tqdm(train_data, total=len(train_data)):
            q_entity = [e.lower() for e in entry["q_entity"]]
            triplets=  [t[1] for t in entry["topk_rel_data"]]
            try:
                if len(q_entity)==0 or len(triplets[0])!=3:
                    count_dirty_data+=1
                if len(triplets)==0 or len(q_entity)==0:
                    graph_feats = torch.zeros((1, 2))
                else:
                    G = nx.DiGraph()
                    for (s, r, o) in triplets:
                        G.add_edge(s.lower(), o.lower(), relation=r.lower())
                    personalization = {n: (1.0 if n in q_entity else 0.0) for n in G.nodes()}
                    ppr_scores = nx.pagerank(
                        G,
                        alpha=0.85,
                        personalization=personalization,
                        max_iter=100,
                        tol=1e-05
                    )
                    graph_feats = []
                    for (s, r, o) in triplets:
                        s_, o_ = s.lower(), o.lower()
                        ppr_s = ppr_scores.get(s_, 0.0)
                        ppr_o = ppr_scores.get(o_, 0.0)
                        graph_feats.append([ppr_s, ppr_o])

                    graph_feats = torch.tensor(graph_feats, dtype=torch.float32)
            except Exception as e:
                print(q_entity)
                print("Triplets: ", triplets)
                print("="*20)
                    
            self.precomputed_data.append({
                "question": entry["question"],
                "is_empty": entry["is_empty"],
                "q_entity": entry["q_entity"],
                "a_entity":entry["a_entity"],
                "answer": entry["answer"],
                "question_embedding": entry["question_embedding"],
                "topk_linearized_triplets": entry["topk_linearized_triplets"],
                "topk_linearized_triplet_embeddings": entry["topk_linearized_triplet_embeddings"],
                "topk_rel_data":entry["topk_rel_data"],
                "topK_rel_embeddings": entry["topK_rel_embeddings"],
                "graph_features": graph_feats.to(self.device)
            })
            
    def __len__(self):
        return len(self.precomputed_data)
    
    def __getitem__(self, idx):
        return self.precomputed_data[idx]

## Path Ranking module

In [ ]:
#NO PPR SCORES
class PathRankingModel(nn.Module):
    def __init__(self, hidden_size=384, device="cuda"):
        super().__init__()
        self.device=device
        self.hidden_size = hidden_size
        self.question_triplet_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size, num_heads=8, batch_first=True, dropout=0.1
        )
        self.question_relation_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size, num_heads=8, batch_first=True, dropout=0.1
        )
        self.gate_network = nn.Sequential(
            nn.Linear(self.hidden_size * 3, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.hidden_size // 2, 1),
            nn.Sigmoid()
        )
        #Triplet centic scorer
        self.triplet_mlp = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

        #  Relation-Centric Scorer
        self.relation_mlp = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

        # Combiner Network
        self.combiner_mlp = nn.Sequential(
            nn.Linear(3, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1) 
        )

        # Temperature and baseline (unchanged)
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)
        self.baseline = nn.Parameter(torch.zeros(1))

    def forward(self, question_embed, triplet_embeds, relation_embeds, graph_scores):
        num_triplets = triplet_embeds.size(0)
        question_embed = question_embed.unsqueeze(0) if question_embed.dim() == 1 else question_embed

        # Existing attention computations (unchanged)
        triplet_attended, triplet_weights = self.question_triplet_attention(
            triplet_embeds, question_embed, question_embed
        )
        relation_attended, relation_weights = self.question_relation_attention(
            relation_embeds, question_embed, question_embed
        )
        triplet_weights = triplet_weights.squeeze(0).squeeze(1)
        relation_weights = relation_weights.squeeze(0).squeeze(1)

        # Gate computation (unchanged)
        question_expanded = question_embed.expand(num_triplets, -1)
        gate_input = torch.cat([question_expanded, triplet_embeds, relation_embeds], dim=-1)
        path_gates = self.gate_network(gate_input).squeeze(-1)
        #avg_ppr_scores = graph_scores.mean(dim=1)
        # Two-Tower Scoring
        # Tower A: Triplet-centric score
        triplet_centric_input = torch.cat([
            triplet_embeds,
            triplet_attended,
            question_expanded,
        ], dim=-1)
        tower_A_scores = self.triplet_mlp(triplet_centric_input).squeeze(-1)
        #print("tower_A_scores: ",tower_A_scores.shape)
        # Tower B: Relation-centric score
        relation_centric_input = torch.cat([
            relation_embeds,
            relation_attended,
            question_expanded,
        ], dim=-1)
        tower_B_scores = self.relation_mlp(relation_centric_input).squeeze(-1)
        #print("tower_B_scores: ",tower_B_scores.shape)
        combiner_input = torch.stack([
            tower_A_scores,
            tower_B_scores,
            path_gates,
        ], dim=-1)
        combined_scores = self.combiner_mlp(combiner_input).squeeze(-1)
        #print("combined_scores: ",combined_scores.shape)
        # Final scoring with temperature
        temp = self.temperature.clamp(min=0.1, max=5.0)
        path_probs = F.softmax(combined_scores / temp, dim=0)

        return combined_scores, path_probs

    
    def sample_paths(self, probabilities: torch.Tensor, paths: List[str], k: int, ranking_scores) -> Tuple[List[str], torch.Tensor, torch.Tensor, torch.Tensor]:
        """Sample k paths using categorical sampling for REINFORCE"""
        # Handle the case where we have fewer paths than k
        if len(paths) <= k:
            # For the case where len(paths) <= k, we need to ensure log_probs has gradients
            log_probs = torch.log(probabilities + 1e-10)  # Add small epsilon to avoid log(0)
            indices = torch.arange(len(paths), device=probabilities.device)
            return paths, probabilities, ranking_scores, log_probs
        dist = torch.distributions.Categorical(probs=probabilities)
        # Sample without replacement
        selected_indices = []
        log_probs_list = []
        remaining_indices = torch.ones(len(probabilities), dtype=torch.bool, device=probabilities.device)
        
        for _ in range(min(k, len(paths))):
            # Create masked probabilities
            masked_probs = probabilities * remaining_indices.float()
            # Re-normalize
            masked_probs = masked_probs / (masked_probs.sum() + 1e-10)
            # Create distribution and sample
            masked_dist = torch.distributions.Categorical(probs=masked_probs)
            idx = masked_dist.sample()
            # Store log probability with gradient
            log_prob = masked_dist.log_prob(idx)
            # Update tracking
            selected_indices.append(idx.item())
            log_probs_list.append(log_prob)
            # Mark as used
            remaining_indices[idx] = False
        
        # Convert indices to tensor
        selected_indices_tensor = torch.tensor(selected_indices, device=probabilities.device)
        
        # Stack log probabilities
        log_probs = torch.stack(log_probs_list)
        
        # Get selected paths
        selected_paths = [paths[i] for i in selected_indices]
        selected_probs = probabilities[selected_indices_tensor]
        selected_ranking_scores = ranking_scores[selected_indices_tensor]
        
        return selected_paths, selected_probs, selected_ranking_scores, log_probs
    
    
    def save_pretrained(self, save_directory: str):
        os.makedirs(save_directory, exist_ok=True)
        path_state = {
            'question_triplet_attention': self.question_triplet_attention.state_dict(),
            'question_relation_attention': self.question_relation_attention.state_dict(),
            "gate_network": self.gate_network.state_dict(),
            "triplet_mlp": self.triplet_mlp.state_dict(),
            "relation_mlp": self.relation_mlp.state_dict(),
            "combiner_mlp": self.combiner_mlp.state_dict(),
            'temperature': self.temperature.detach().cpu(),
            'baseline': self.baseline.detach().cpu()
        }
        torch.save(path_state, os.path.join(save_directory, "path_ranker.pt"))
    
    @classmethod
    def from_pretrained(cls, load_directory: str):
        """Load model using HuggingFace from_pretrained"""
        model = cls()
        extra_state = torch.load(os.path.join(load_directory, "path_ranker.pt"))
        model.question_triplet_attention.load_state_dict(extra_state['question_triplet_attention'])
        model.question_relation_attention.load_state_dict(extra_state['question_relation_attention'])
        model.gate_network.load_state_dict(extra_state['gate_network'])
        model.triplet_mlp.load_state_dict(extra_state['triplet_mlp'])
        model.relation_mlp.load_state_dict(extra_state['relation_mlp'])
        model.combiner_mlp.load_state_dict(extra_state['combiner_mlp'])
        model.temperature.data = extra_state['temperature'].to(model.device)
        model.baseline.data = extra_state['baseline'].to(model.device)
        return model

In [ ]:
# #NO Gate SCORES
# class PathRankingModel(nn.Module):
#     def __init__(self, hidden_size=384, device="cuda"):
#         super().__init__()
#         self.device=device
#         self.hidden_size = hidden_size
#         self.question_triplet_attention = nn.MultiheadAttention(
#             embed_dim=self.hidden_size, num_heads=8, batch_first=True, dropout=0.1
#         )
#         self.question_relation_attention = nn.MultiheadAttention(
#             embed_dim=self.hidden_size, num_heads=8, batch_first=True, dropout=0.1
#         )
# #         self.gate_network = nn.Sequential(
# #             nn.Linear(self.hidden_size * 3, self.hidden_size),
# #             nn.LayerNorm(self.hidden_size),
# #             nn.ReLU(),
# #             nn.Dropout(0.1),
# #             nn.Linear(self.hidden_size, self.hidden_size // 2),
# #             nn.ReLU(),
# #             nn.Linear(self.hidden_size // 2, 1),
# #             nn.Sigmoid()
# #         )
#         #Triplet centic scorer
#         self.triplet_mlp = nn.Sequential(
#             nn.Linear(hidden_size * 3+2, hidden_size),
#             nn.LayerNorm(hidden_size),
#             nn.ReLU(),
#             nn.Dropout(0.1),
#             nn.Linear(hidden_size, hidden_size // 2),
#             nn.ReLU(),
#             nn.Linear(hidden_size // 2, 1)
#         )

#         #  Relation-Centric Scorer
#         self.relation_mlp = nn.Sequential(
#             nn.Linear(hidden_size * 3+2, hidden_size),
#             nn.LayerNorm(hidden_size),
#             nn.ReLU(),
#             nn.Dropout(0.1),
#             nn.Linear(hidden_size, hidden_size // 2),
#             nn.ReLU(),
#             nn.Linear(hidden_size // 2, 1)
#         )

#         # Combiner Network
#         self.combiner_mlp = nn.Sequential(
#             nn.Linear(2, hidden_size // 2),
#             nn.ReLU(),
#             nn.Dropout(0.1),
#             nn.Linear(hidden_size // 2, 1) 
#         )

#         # Temperature and baseline (unchanged)
#         self.temperature = nn.Parameter(torch.ones(1) * 1.0)
#         self.baseline = nn.Parameter(torch.zeros(1))

#     def forward(self, question_embed, triplet_embeds, relation_embeds, graph_scores):
#         num_triplets = triplet_embeds.size(0)
#         question_embed = question_embed.unsqueeze(0) if question_embed.dim() == 1 else question_embed

#         # Existing attention computations (unchanged)
#         triplet_attended, triplet_weights = self.question_triplet_attention(
#             triplet_embeds, question_embed, question_embed
#         )
#         relation_attended, relation_weights = self.question_relation_attention(
#             relation_embeds, question_embed, question_embed
#         )
#         triplet_weights = triplet_weights.squeeze(0).squeeze(1)
#         relation_weights = relation_weights.squeeze(0).squeeze(1)

#         # Gate computation (unchanged)
#         question_expanded = question_embed.expand(num_triplets, -1)
# #         gate_input = torch.cat([question_expanded, triplet_embeds, relation_embeds], dim=-1)
# #         path_gates = self.gate_network(gate_input).squeeze(-1)
#         #avg_ppr_scores = graph_scores.mean(dim=1)
#         # Two-Tower Scoring
#         # Tower A: Triplet-centric score
#         triplet_centric_input = torch.cat([
#             triplet_embeds,
#             triplet_attended,
#             question_expanded,
#             graph_scores
#         ], dim=-1)
#         tower_A_scores = self.triplet_mlp(triplet_centric_input).squeeze(-1)
#         #print("tower_A_scores: ",tower_A_scores.shape)
#         # Tower B: Relation-centric score
#         relation_centric_input = torch.cat([
#             relation_embeds,
#             relation_attended,
#             question_expanded,
#             graph_scores
#         ], dim=-1)
#         tower_B_scores = self.relation_mlp(relation_centric_input).squeeze(-1)
#         #print("tower_B_scores: ",tower_B_scores.shape)
#         combiner_input = torch.stack([
#             tower_A_scores,
#             tower_B_scores,
#             #path_gates,
#         ], dim=-1)
#         combined_scores = self.combiner_mlp(combiner_input).squeeze(-1)
#         #print("combined_scores: ",combined_scores.shape)
#         # Final scoring with temperature
#         temp = self.temperature.clamp(min=0.1, max=5.0)
#         path_probs = F.softmax(combined_scores / temp, dim=0)

#         return combined_scores, path_probs

    
#     def sample_paths(self, probabilities: torch.Tensor, paths: List[str], k: int, ranking_scores) -> Tuple[List[str], torch.Tensor, torch.Tensor, torch.Tensor]:
#         """Sample k paths using categorical sampling for REINFORCE"""
#         # Handle the case where we have fewer paths than k
#         if len(paths) <= k:
#             # For the case where len(paths) <= k, we need to ensure log_probs has gradients
#             log_probs = torch.log(probabilities + 1e-10)  # Add small epsilon to avoid log(0)
#             indices = torch.arange(len(paths), device=probabilities.device)
#             return paths, probabilities, ranking_scores, log_probs
#         dist = torch.distributions.Categorical(probs=probabilities)
#         # Sample without replacement
#         selected_indices = []
#         log_probs_list = []
#         remaining_indices = torch.ones(len(probabilities), dtype=torch.bool, device=probabilities.device)
        
#         for _ in range(min(k, len(paths))):
#             # Create masked probabilities
#             masked_probs = probabilities * remaining_indices.float()
#             # Re-normalize
#             masked_probs = masked_probs / (masked_probs.sum() + 1e-10)
#             # Create distribution and sample
#             masked_dist = torch.distributions.Categorical(probs=masked_probs)
#             idx = masked_dist.sample()
#             # Store log probability with gradient
#             log_prob = masked_dist.log_prob(idx)
#             # Update tracking
#             selected_indices.append(idx.item())
#             log_probs_list.append(log_prob)
#             # Mark as used
#             remaining_indices[idx] = False
        
#         # Convert indices to tensor
#         selected_indices_tensor = torch.tensor(selected_indices, device=probabilities.device)
        
#         # Stack log probabilities
#         log_probs = torch.stack(log_probs_list)
        
#         # Get selected paths
#         selected_paths = [paths[i] for i in selected_indices]
#         selected_probs = probabilities[selected_indices_tensor]
#         selected_ranking_scores = ranking_scores[selected_indices_tensor]
        
#         return selected_paths, selected_probs, selected_ranking_scores, log_probs
    
    
#     def save_pretrained(self, save_directory: str):
#         os.makedirs(save_directory, exist_ok=True)
#         path_state = {
#             'question_triplet_attention': self.question_triplet_attention.state_dict(),
#             'question_relation_attention': self.question_relation_attention.state_dict(),
#             #"gate_network": self.gate_network.state_dict(),
#             "triplet_mlp": self.triplet_mlp.state_dict(),
#             "relation_mlp": self.relation_mlp.state_dict(),
#             "combiner_mlp": self.combiner_mlp.state_dict(),
#             'temperature': self.temperature.detach().cpu(),
#             'baseline': self.baseline.detach().cpu()
#         }
#         torch.save(path_state, os.path.join(save_directory, "path_ranker.pt"))
    
#     @classmethod
#     def from_pretrained(cls, load_directory: str):
#         """Load model using HuggingFace from_pretrained"""
#         model = cls()
#         extra_state = torch.load(os.path.join(load_directory, "path_ranker.pt"))
#         model.question_triplet_attention.load_state_dict(extra_state['question_triplet_attention'])
#         model.question_relation_attention.load_state_dict(extra_state['question_relation_attention'])
#         #model.gate_network.load_state_dict(extra_state['gate_network'])
#         model.triplet_mlp.load_state_dict(extra_state['triplet_mlp'])
#         model.relation_mlp.load_state_dict(extra_state['relation_mlp'])
#         model.combiner_mlp.load_state_dict(extra_state['combiner_mlp'])
#         model.temperature.data = extra_state['temperature'].to(model.device)
#         model.baseline.data = extra_state['baseline'].to(model.device)
#         return model

In [ ]:
# class PathRankingModel(nn.Module):
#     def __init__(self, hidden_size=384, device="cuda"):
#         super().__init__()
#         self.device=device
#         self.hidden_size = hidden_size
#         self.question_triplet_attention = nn.MultiheadAttention(
#             embed_dim=self.hidden_size, num_heads=8, batch_first=True, dropout=0.1
#         )
#         self.question_relation_attention = nn.MultiheadAttention(
#             embed_dim=self.hidden_size, num_heads=8, batch_first=True, dropout=0.1
#         )
#         self.gate_network = nn.Sequential(
#             nn.Linear(self.hidden_size * 3, self.hidden_size),
#             nn.LayerNorm(self.hidden_size),
#             nn.ReLU(),
#             nn.Dropout(0.1),
#             nn.Linear(self.hidden_size, self.hidden_size // 2),
#             nn.ReLU(),
#             nn.Linear(self.hidden_size // 2, 1),
#             nn.Sigmoid()
#         )
#         #Triplet centic scorer
#         self.triplet_mlp = nn.Sequential(
#             nn.Linear(hidden_size * 3+2, hidden_size),
#             nn.LayerNorm(hidden_size),
#             nn.ReLU(),
#             nn.Dropout(0.1),
#             nn.Linear(hidden_size, hidden_size // 2),
#             nn.ReLU(),
#             nn.Linear(hidden_size // 2, 1)
#         )

#         #  Relation-Centric Scorer
#         self.relation_mlp = nn.Sequential(
#             nn.Linear(hidden_size * 3+2, hidden_size),
#             nn.LayerNorm(hidden_size),
#             nn.ReLU(),
#             nn.Dropout(0.1),
#             nn.Linear(hidden_size, hidden_size // 2),
#             nn.ReLU(),
#             nn.Linear(hidden_size // 2, 1)
#         )

#         # Combiner Network
#         self.combiner_mlp = nn.Sequential(
#             nn.Linear(3, hidden_size // 2),  # Input: [tower_A_score, tower_B_score, graph_feats, attention_delta]
#             nn.ReLU(),
#             nn.Dropout(0.1),
#             nn.Linear(hidden_size // 2, 1)   # Final score adjustment
#         )

#         # Temperature and baseline (unchanged)
#         self.temperature = nn.Parameter(torch.ones(1) * 1.0)
#         self.baseline = nn.Parameter(torch.zeros(1))

#     def forward(self, question_embed, triplet_embeds, relation_embeds, graph_scores):
#         num_triplets = triplet_embeds.size(0)
#         question_embed = question_embed.unsqueeze(0) if question_embed.dim() == 1 else question_embed

#         # Existing attention computations (unchanged)
#         triplet_attended, triplet_weights = self.question_triplet_attention(
#             triplet_embeds, question_embed, question_embed
#         )
#         relation_attended, relation_weights = self.question_relation_attention(
#             relation_embeds, question_embed, question_embed
#         )
#         triplet_weights = triplet_weights.squeeze(0).squeeze(1)
#         relation_weights = relation_weights.squeeze(0).squeeze(1)

#         # Gate computation (unchanged)
#         question_expanded = question_embed.expand(num_triplets, -1)
#         gate_input = torch.cat([question_expanded, triplet_embeds, relation_embeds], dim=-1)
#         path_gates = self.gate_network(gate_input).squeeze(-1)
#         #avg_ppr_scores = graph_scores.mean(dim=1)
#         # Two-Tower Scoring
#         # Tower A: Triplet-centric score
#         triplet_centric_input = torch.cat([
#             triplet_embeds,
#             triplet_attended,
#             question_expanded,
#             graph_scores
#         ], dim=-1)
#         tower_A_scores = self.triplet_mlp(triplet_centric_input).squeeze(-1)
#         #print("tower_A_scores: ",tower_A_scores.shape)
#         # Tower B: Relation-centric score
#         relation_centric_input = torch.cat([
#             relation_embeds,
#             relation_attended,
#             question_expanded,
#             graph_scores
#         ], dim=-1)
#         tower_B_scores = self.relation_mlp(relation_centric_input).squeeze(-1)
#         #print("tower_B_scores: ",tower_B_scores.shape)
#         combiner_input = torch.stack([
#             tower_A_scores,
#             tower_B_scores,
#             path_gates,
#         ], dim=-1)
#         combined_scores = self.combiner_mlp(combiner_input).squeeze(-1)
#         #print("combined_scores: ",combined_scores.shape)
#         # Final scoring with temperature
#         temp = self.temperature.clamp(min=0.1, max=5.0)
#         path_probs = F.softmax(combined_scores / temp, dim=0)

#         return combined_scores, path_probs

    
#     def sample_paths(self, probabilities: torch.Tensor, paths: List[str], k: int, ranking_scores) -> Tuple[List[str], torch.Tensor, torch.Tensor, torch.Tensor]:
#         """Sample k paths using categorical sampling for REINFORCE"""
#         # Handle the case where we have fewer paths than k
#         if len(paths) <= k:
#             # For the case where len(paths) <= k, we need to ensure log_probs has gradients
#             log_probs = torch.log(probabilities + 1e-10)  # Add small epsilon to avoid log(0)
#             indices = torch.arange(len(paths), device=probabilities.device)
#             return paths, probabilities, ranking_scores, log_probs
#         dist = torch.distributions.Categorical(probs=probabilities)
#         # Sample without replacement
#         selected_indices = []
#         log_probs_list = []
#         remaining_indices = torch.ones(len(probabilities), dtype=torch.bool, device=probabilities.device)
        
#         for _ in range(min(k, len(paths))):
#             # Create masked probabilities
#             masked_probs = probabilities * remaining_indices.float()
#             # Re-normalize
#             masked_probs = masked_probs / (masked_probs.sum() + 1e-10)
#             # Create distribution and sample
#             masked_dist = torch.distributions.Categorical(probs=masked_probs)
#             idx = masked_dist.sample()
#             # Store log probability with gradient
#             log_prob = masked_dist.log_prob(idx)
#             # Update tracking
#             selected_indices.append(idx.item())
#             log_probs_list.append(log_prob)
#             # Mark as used
#             remaining_indices[idx] = False
        
#         # Convert indices to tensor
#         selected_indices_tensor = torch.tensor(selected_indices, device=probabilities.device)
        
#         # Stack log probabilities
#         log_probs = torch.stack(log_probs_list)
        
#         # Get selected paths
#         selected_paths = [paths[i] for i in selected_indices]
#         selected_probs = probabilities[selected_indices_tensor]
#         selected_ranking_scores = ranking_scores[selected_indices_tensor]
        
#         return selected_paths, selected_probs, selected_ranking_scores, log_probs
    
    
#     def save_pretrained(self, save_directory: str):
#         os.makedirs(save_directory, exist_ok=True)
#         path_state = {
#             'question_triplet_attention': self.question_triplet_attention.state_dict(),
#             'question_relation_attention': self.question_relation_attention.state_dict(),
#             "gate_network": self.gate_network.state_dict(),
#             "triplet_mlp": self.triplet_mlp.state_dict(),
#             "relation_mlp": self.relation_mlp.state_dict(),
#             "combiner_mlp": self.combiner_mlp.state_dict(),
#             'temperature': self.temperature.detach().cpu(),
#             'baseline': self.baseline.detach().cpu()
#         }
#         torch.save(path_state, os.path.join(save_directory, "path_ranker.pt"))
    
#     @classmethod
#     def from_pretrained(cls, load_directory: str):
#         """Load model using HuggingFace from_pretrained"""
#         model = cls()
#         extra_state = torch.load(os.path.join(load_directory, "path_ranker.pt"))
#         model.question_triplet_attention.load_state_dict(extra_state['question_triplet_attention'])
#         model.question_relation_attention.load_state_dict(extra_state['question_relation_attention'])
#         model.gate_network.load_state_dict(extra_state['gate_network'])
#         model.triplet_mlp.load_state_dict(extra_state['triplet_mlp'])
#         model.relation_mlp.load_state_dict(extra_state['relation_mlp'])
#         model.combiner_mlp.load_state_dict(extra_state['combiner_mlp'])
#         model.temperature.data = extra_state['temperature'].to(model.device)
#         model.baseline.data = extra_state['baseline'].to(model.device)
#         return model

In [ ]:
class JointTrainingDatasetv3PPR(Dataset):
    def __init__(self, train_data, device='cpu'):
        self.device = device
        # This dataset is prepared using  JointTrainingDatasetv3 dataset. Only graph features are computed in this dataset
        self.precomputed_data = []
        for entry in tqdm(train_data, total=len(train_data)):
            q_entity = [e.lower() for e in entry["q_entity"]]
            triplets=  [t[1] for t in entry["topk_rel_data"]]
            try:
                if len(q_entity)==0 or len(triplets[0])!=3:
                    count_dirty_data+=1
                if len(triplets)==0 or len(q_entity)==0:
                    graph_feats = torch.zeros((1, 2))
                else:
                    G = nx.DiGraph()
                    for (s, r, o) in triplets:
                        G.add_edge(s.lower(), o.lower(), relation=r.lower())
                    personalization = {n: (1.0 if n in q_entity else 0.0) for n in G.nodes()}
                    ppr_scores = nx.pagerank(
                        G,
                        alpha=0.85,
                        personalization=personalization,
                        max_iter=100,
                        tol=1e-05
                    )
                    graph_feats = []
                    for (s, r, o) in triplets:
                        s_, o_ = s.lower(), o.lower()
                        ppr_s = ppr_scores.get(s_, 0.0)
                        ppr_o = ppr_scores.get(o_, 0.0)
                        graph_feats.append([ppr_s, ppr_o])

                    graph_feats = torch.tensor(graph_feats, dtype=torch.float32)
            except Exception as e:
                print(q_entity)
                print("Triplets: ", triplets)
                print("="*20)
                    
            self.precomputed_data.append({
                "question": entry["question"],
                "is_empty": entry["is_empty"],
                "q_entity": entry["q_entity"],
                "a_entity":entry["a_entity"],
                "answer": entry["answer"],
                "question_embedding": entry["question_embedding"].to(self.device),
                "topk_linearized_triplets": entry["topk_linearized_triplets"],
                "topk_linearized_triplet_embeddings": entry["topk_linearized_triplet_embeddings"].to(self.device),
                "topk_rel_data":entry["topk_rel_data"],
                "topK_rel_embeddings": entry["topK_rel_embeddings"].to(self.device),
                "graph_features": graph_feats.to(self.device)
            })
            
    def __len__(self):
        return len(self.precomputed_data)
    
    def __getitem__(self, idx):
        return self.precomputed_data[idx]

## Warmup Module

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from typing import Dict, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import ndcg_score
import os

class SelectorPretrainer(nn.Module):
    def __init__(self, path_ranker):
        super().__init__()
        self.ranker = path_ranker
        
    def forward(self, ques_embed, linearized_triplet_embd, rel_embd, graph_scores):
        raw_scores, _ = self.ranker(ques_embed, linearized_triplet_embd, rel_embd, graph_scores )
        return raw_scores


class CosinePretrainingDataset:
    def __init__(self, original_dataset, k=500):
        self.original_dataset = original_dataset
        self.k=k
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        data = self.original_dataset[idx]
        
        # Skip empty path samples during pretraining
        if len(data["topk_linearized_triplets"])==0 or len(data["q_entity"])==0:
            return None
    
        question_embedding = data["question_embedding"]  # [embedding_dim]
        if data["topk_linearized_triplet_embeddings"].shape[0] >=self.k:
            use_nums = self.k
        else:
            use_nums = data["topk_linearized_triplet_embeddings"].shape[0]
        path_embeddings = data["topk_linearized_triplet_embeddings"][:use_nums]
        rel_embeddings = data["topK_rel_embeddings"][:use_nums]  
        num_paths = use_nums
        cosine_targets = self.create_decreasing_targets(num_paths)
        return {
            "question_embedding": question_embedding,
            "path_embeddings": path_embeddings,
            "rel_embeddings": rel_embeddings,
            "cosine_targets": cosine_targets,
            "question": data["question"],
            "paths": data["topk_linearized_triplets"][:use_nums],
            "num_paths": num_paths,
            "graph_features":data["graph_features"][:use_nums]
        }
    
    def create_decreasing_targets(self, num_paths):
        """Create target scores that decrease with rank (since paths are pre-sorted by cosine similarity)"""
        # Linear decay
        #targets = torch.linspace(1.0, 0.0, num_paths)
        
        #Exponential decay
        decay_rate = 0.01
        targets = torch.exp(-decay_rate * torch.arange(num_paths, dtype=torch.float))
        
        #Log-based decay
        #targets = 1.0 / torch.log(2 + torch.arange(num_paths, dtype=torch.float))
        
        return targets


def collate_fn_pretrain(batch):
    """Custom collate function that filters out None samples"""
    # Filter out None samples
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    return batch[0]



class CosinePretrainer:
    def __init__(
        self, 
        path_ranker, 
        device="cuda",
        checkpoint_dir="pretrain_checkpoints"
    ):
        self.device = device
        self.pretrainer = SelectorPretrainer(path_ranker).to(device)
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        # Loss functions
        self.mse_loss = nn.MSELoss()
        self.ranking_loss = nn.MarginRankingLoss(margin=0.1)
        
        # Metrics tracking
        self.train_losses = []
        self.val_losses = []
        self.correlations = []
        self.ndcg_scores = []
        
    def compute_ranking_loss(self, predicted_scores, target_scores, sample_pairs=100):
        """Compute pairwise ranking loss with sampling for efficiency"""
        batch_size = predicted_scores.size(0)
        if batch_size < 2:
            return torch.tensor(0.0, device=self.device, requires_grad=True)
        
        # Sample pairs for efficiency (especially important for 500 paths)
        max_pairs = min(sample_pairs, (batch_size * (batch_size - 1)) // 2)
        
        # Randomly sample pairs
        all_pairs = []
        for i in range(batch_size):
            for j in range(i + 1, batch_size):
                all_pairs.append((i, j))
        
        if len(all_pairs) > max_pairs:
            sampled_pairs = np.random.choice(len(all_pairs), max_pairs, replace=False)
            pairs = [all_pairs[i] for i in sampled_pairs]
        else:
            pairs = all_pairs
        
        if len(pairs) == 0:
            return torch.tensor(0.0, device=self.device, requires_grad=True)
        
        pairs_i = torch.tensor([p[0] for p in pairs], device=self.device)
        pairs_j = torch.tensor([p[1] for p in pairs], device=self.device)
        
        # Create targets: 1 if target_scores[i] > target_scores[j], -1 otherwise
        targets = torch.sign(target_scores[pairs_i] - target_scores[pairs_j])
        
        pred_i = predicted_scores[pairs_i]
        pred_j = predicted_scores[pairs_j]
        
        return self.ranking_loss(pred_i, pred_j, targets)
    
    def compute_combined_loss(self, predicted_scores, target_scores, alpha=0.5):
        """Combine MSE and ranking loss"""
        mse = self.mse_loss(predicted_scores, target_scores)
        ranking = self.compute_ranking_loss(predicted_scores, target_scores)
        
        combined_loss = alpha * mse + (1 - alpha) * ranking
        return combined_loss, mse.item(), ranking.item()
    
    def compute_spearman_correlation(self, predicted_scores, target_scores):
        """Compute Spearman correlation between predicted and target rankings"""
        pred_np = predicted_scores.detach().cpu().numpy()
        target_np = target_scores.detach().cpu().numpy()
        
        # Compute ranks
        pred_ranks = np.argsort(np.argsort(-pred_np))  # Higher scores get lower ranks
        target_ranks = np.argsort(np.argsort(-target_np))
        
        # Spearman correlation
        correlation = np.corrcoef(pred_ranks, target_ranks)[0, 1]
        return correlation if not np.isnan(correlation) else 0.0
    
    def compute_ndcg_at_k(self, predicted_scores, target_scores, k=50):
        """Compute NDCG@k to measure ranking quality"""
        pred_np = predicted_scores.detach().cpu().numpy().reshape(1, -1)
        target_np = target_scores.detach().cpu().numpy().reshape(1, -1)
        
        try:
            ndcg = ndcg_score(target_np, pred_np, k=min(k, len(target_np[0])))
            return ndcg
        except:
            return 0.0
    
    def train_step(self, batch):
        if batch is None:
            return None, None, None, None, None
            
        question_embed = batch["question_embedding"].to(self.device)
        path_embeds = batch["path_embeddings"].to(self.device).to(self.device)
        rel_embeds = batch["rel_embeddings"].to(self.device)
        cosine_targets = batch["cosine_targets"].to(self.device)
        graph_features = batch["graph_features"].to(self.device)
        predicted_scores = self.pretrainer(question_embed.unsqueeze(0), path_embeds, rel_embeds, graph_features)
        combined_loss, mse_loss, ranking_loss = self.compute_combined_loss(
            predicted_scores, cosine_targets
        )
        correlation = self.compute_spearman_correlation(predicted_scores, cosine_targets)
        ndcg = self.compute_ndcg_at_k(predicted_scores, cosine_targets, k=50)
        
        return combined_loss, mse_loss, ranking_loss, correlation, ndcg
    
    @torch.no_grad()
    def validate(self, val_dataloader):
        """Validation loop"""
        self.pretrainer.eval()
        
        total_loss = 0
        total_mse = 0
        total_ranking = 0
        total_correlation = 0
        total_ndcg = 0
        valid_samples = 0
        
        for batch in tqdm(val_dataloader, desc="Validation"):
            loss, mse, ranking, corr, ndcg = self.train_step(batch)
            if loss is None:
                continue
                
            total_loss += loss.item()
            total_mse += mse
            total_ranking += ranking
            total_correlation += corr
            total_ndcg += ndcg
            valid_samples += 1
        
        if valid_samples == 0:
            return 0, 0, 0, 0, 0
            
        return (
            total_loss / valid_samples,
            total_mse / valid_samples,
            total_ranking / valid_samples,
            total_correlation / valid_samples,
            total_ndcg / valid_samples
        )
    
    def train(self, train_dataloader, val_dataloader=None, num_epochs=5, learning_rate=1e-4,
            weight_decay=1e-5, gradient_accumulation_steps=8, validation_interval=1, save_best=True):
        
        optimizer = torch.optim.AdamW(
            self.pretrainer.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=2, factor=0.5, verbose=True
        )
        
        best_val_loss = float('inf')
        best_correlation = -1.0
        
        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch + 1}/{num_epochs}")
            
            # Training
            self.pretrainer.train()
            epoch_loss = 0
            epoch_mse = 0
            epoch_ranking = 0
            epoch_correlation = 0
            epoch_ndcg = 0
            valid_batches = 0
            
            optimizer.zero_grad()
            
            with tqdm(train_dataloader, desc="Training") as pbar:
                for batch_idx, batch in enumerate(pbar):
                    loss, mse, ranking, corr, ndcg = self.train_step(batch)
                    if loss is None:
                        continue
                    scaled_loss = loss / gradient_accumulation_steps
                    scaled_loss.backward()
                    
                    # Update metrics
                    epoch_loss += loss.item()
                    epoch_mse += mse
                    epoch_ranking += ranking
                    epoch_correlation += corr
                    epoch_ndcg += ndcg
                    valid_batches += 1
                    if (batch_idx + 1) % gradient_accumulation_steps == 0:
                        torch.nn.utils.clip_grad_norm_(self.pretrainer.parameters(), 1.0)
                        optimizer.step()
                        optimizer.zero_grad()
                    if valid_batches > 0:
                        pbar.set_postfix({
                            'loss': f"{epoch_loss/valid_batches:.4f}",
                            'corr': f"{epoch_correlation/valid_batches:.3f}",
                            'ndcg': f"{epoch_ndcg/valid_batches:.3f}"
                        })
            if valid_batches % gradient_accumulation_steps != 0:
                optimizer.step()
                optimizer.zero_grad()
            
            # Calculate epoch averages
            if valid_batches > 0:
                avg_train_loss = epoch_loss / valid_batches
                avg_train_corr = epoch_correlation / valid_batches
                avg_train_ndcg = epoch_ndcg / valid_batches
                
                self.train_losses.append(avg_train_loss)
                print(f"Train - Loss: {avg_train_loss:.4f}, Correlation: {avg_train_corr:.3f}, NDCG@50: {avg_train_ndcg:.3f}")
            
            # Validation
            if val_dataloader and (epoch + 1) % validation_interval == 0:
                val_loss, val_mse, val_ranking, val_corr, val_ndcg = self.validate(val_dataloader)
                self.val_losses.append(val_loss)
                self.correlations.append(val_corr)
                self.ndcg_scores.append(val_ndcg)
                
                print(f"Val - Loss: {val_loss:.4f}, Correlation: {val_corr:.3f}, NDCG@50: {val_ndcg:.3f}")
                scheduler.step(val_loss)
                if save_best and (val_corr > best_correlation or val_loss < best_val_loss):
                    if val_corr > best_correlation:
                        best_correlation = val_corr
                        print(f"New best correlation: {best_correlation:.3f}")
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        print(f"New best validation loss: {best_val_loss:.4f}")
                    
                    self.save_checkpoint(epoch + 1, val_loss, val_corr)
        self.plot_training_progress()
        print("Pretraining completed!")
    
    def save_checkpoint(self, epoch, val_loss, val_corr):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.pretrainer.ranker.state_dict(),
            'val_loss': val_loss,
            'val_correlation': val_corr,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'correlations': self.correlations,
            'ndcg_scores': self.ndcg_scores
        }
        
        torch.save(checkpoint, os.path.join(self.checkpoint_dir, f'best_pretrained_model-{epoch}.pt'))
        print(f"Checkpoint saved at epoch {epoch}")
    
    def load_checkpoint(self, checkpoint_path):
        """Load model checkpoint"""
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.pretrainer.ranker.load_state_dict(checkpoint['model_state_dict'])
        self.train_losses = checkpoint.get('train_losses', [])
        self.val_losses = checkpoint.get('val_losses', [])
        self.correlations = checkpoint.get('correlations', [])
        self.ndcg_scores = checkpoint.get('ndcg_scores', [])
        print(f"Checkpoint loaded. Best correlation: {checkpoint['val_correlation']:.3f}")
    
    def plot_training_progress(self):
        """Plot training metrics"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Loss curves
        axes[0, 0].plot(self.train_losses, label='Train Loss', alpha=0.7)
        axes[0, 0].plot(self.val_losses, label='Val Loss', alpha=0.7)
        axes[0, 0].set_title('Training and Validation Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Correlation
        axes[0, 1].plot(self.correlations, label='Spearman Correlation', color='green', alpha=0.7)
        axes[0, 1].set_title('Validation Spearman Correlation')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Correlation')
        axes[0, 1].grid(True, alpha=0.3)
        axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
        
        # NDCG scores
        axes[1, 0].plot(self.ndcg_scores, label='NDCG@50', color='orange', alpha=0.7)
        axes[1, 0].set_title('Validation NDCG@50')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('NDCG@50')
        axes[1, 0].grid(True, alpha=0.3)
        
        # Combined metrics
        if len(self.correlations) > 0 and len(self.ndcg_scores) > 0:
            epochs = range(1, len(self.correlations) + 1)
            ax2 = axes[1, 1]
            ax3 = ax2.twinx()
            
            line1 = ax2.plot(epochs, self.correlations, 'g-', alpha=0.7, label='Correlation')
            line2 = ax3.plot(epochs, self.ndcg_scores, 'orange', alpha=0.7, label='NDCG@50')
            
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Spearman Correlation', color='g')
            ax3.set_ylabel('NDCG@50', color='orange')
            ax2.set_title('Correlation vs NDCG@50')
            
            # Combine legends
            lines = line1 + line2
            labels = [l.get_label() for l in lines]
            ax2.legend(lines, labels, loc='upper left')
            
            ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.checkpoint_dir, 'pretraining_progress.png'), dpi=300, bbox_inches='tight')
        plt.show()

# Usage example:
def run_pretraining(train_dataset, val_dataset, path_ranker, device="cuda", checkpoint_dir = ""):
    """
    Main function to run pretraining
    
    Args:
        train_dataset: JointTrainingDataset for training
        val_dataset: JointTrainingDataset for validation  
        path_ranker: PathRankingModel to pretrain
        device: Device to run on
    """
    
    # Create pretraining datasets
    pretrain_train_dataset = CosinePretrainingDataset(train_dataset)
    pretrain_val_dataset = CosinePretrainingDataset(val_dataset) if val_dataset else None
    
    # Create dataloaders
    train_dataloader = DataLoader(
        pretrain_train_dataset,
        batch_size=1,  # Process one question at a time
        shuffle=True,
        collate_fn=collate_fn_pretrain,
        num_workers=0  # Set to 0 to avoid multiprocessing issues
    )
    
    val_dataloader = None
    if pretrain_val_dataset:
        val_dataloader = DataLoader(
            pretrain_val_dataset,
            batch_size=1,
            shuffle=False,
            collate_fn=collate_fn_pretrain,
            num_workers=0
        )
    
    pretrainer = CosinePretrainer(path_ranker, device=device, checkpoint_dir = checkpoint_dir)
    
    pretrainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        num_epochs=5,
        learning_rate=1e-4,
        gradient_accumulation_steps=32
    )
    
    return pretrainer

In [ ]:
# ptrain_dataset = torch.load("/mnt/LS226/LS25/sourav23099/webqsp/webqsp-v2/train/train_jointrainer_dataset_pretrain_ppr.pt", weights_only=False)
# pval_dataset = torch.load("/mnt/LS226/LS25/sourav23099/webqsp/webqsp-v2/val/val_jointrainer_dataset_pretrain_ppr.pt", weights_only=False)

In [ ]:
# def move_dataset_to_cpu(dataset):
#     for entry in dataset.precomputed_data:
#         for key in ["question_embedding", "topk_linearized_triplet_embeddings", "topK_rel_embeddings", "graph_features"]:
#             if isinstance(entry.get(key), torch.Tensor):
#                 entry[key] = entry[key].cpu()
# move_dataset_to_cpu(pval_dataset)

In [ ]:
# import gc
# import torch

# # Manually trigger garbage collection
# gc.collect()

# # Clear GPU memory cache
# torch.cuda.empty_cache()
# print("done")


In [ ]:
ptrain_dataset = torch.load("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/train/train_jointrainer_path_dataset_v3_ppr.pt", weights_only=False, map_location="cpu")
pval_dataset = torch.load("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/val/val_jointrainer_path_dataset_v3_ppr.pt", weights_only=False,  map_location="cpu")

In [ ]:
# sbert =  SentenceTransformer('all-MiniLM-L6-v2')
# ppr_train_dataset = JointTrainingDatasetv3PPRv3(ptrain_dataset)
# ppr_val_dataset = JointTrainingDatasetv3PPRv3(pval_dataset)

In [ ]:
path_ranker = PathRankingModel()

base_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/path_ranker_pretrained/architecture-v8/v1-n500_cosine_no_gate"

pretrainer = run_pretraining(
    train_dataset=ptrain_dataset,
    val_dataset=pval_dataset, 
    path_ranker=path_ranker,
    device="cuda",
    checkpoint_dir =base_dir
) 


# Joint Trainer

In [ ]:
from tqdm import tqdm
import unicodedata
from difflib import SequenceMatcher
import Levenshtein

reward_error_list= []
count_warning = 0
count_error = 0
class JointTrainer:
    def __init__(
        self,
        path_ranker: PathRankingModel,
        reward_func,
        device: str = "cuda",
        max_grad_norm: float = 1.0,
        gradient_accumulation_steps: int = 16,
        checkpoint_dir: str = "checkpoints",
        gamma=0.99,
        baseline_decay=0.9
    ):
        self.reward_func = reward_func
        self.path_ranker = path_ranker.to(device)
        self.device = device
        self.max_grad_norm = max_grad_norm
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.gamma = gamma
        self.baseline_decay = baseline_decay
        self.running_baseline = 0
        # Add reward buffer for accumulation
        self.reward_buffer = []
        # Track best validation loss
        self.best_val_reward = float('-inf')
        self.device = device
        
    def compute_reinforce_loss(self, log_probs, rewards, baseline):
        """
        Compute REINFORCE loss with baseline for variance reduction
        
        Args:
            log_probs: Log probabilities of selected actions (paths)
            rewards: Rewards for selected actions (negative LLM loss)
            baseline: Optional baseline value for variance reduction
            
        Returns:
            REINFORCE loss
        """
        # If no explicit baseline is provided, use the running average
        if baseline is None:
            baseline = self.path_ranker.baseline.detach()
            print("baseline was none: ", baseline)
        # Calculate advantages
        advantages = rewards - baseline
        # REINFORCE loss is negative expected reward
        # We're maximizing expected reward, so we negate it for gradient descent
        reinforce_loss = -(log_probs * advantages.detach()).mean()
        return reinforce_loss

    
    def update_baseline_with_buffer(self):
        if len(self.reward_buffer) > 0:
            avg_reward = sum(self.reward_buffer) / len(self.reward_buffer)

            # Much more conservative baseline updates
            if self.running_baseline == 0:
                self.running_baseline = avg_reward * 0.8  # Start below actual rewards
            else:
                # Only update if we're significantly off, and do it slowly
                error = avg_reward - self.running_baseline
                if abs(error) > 0.5:  # Only update for significant differences
                    self.running_baseline += 0.1 * error  # Very slow updates

            # Cap baseline to prevent overshoot
            max_reasonable_baseline = avg_reward * 0.9
            self.running_baseline = min(self.running_baseline, max_reasonable_baseline)

            self.path_ranker.baseline.data = torch.tensor([self.running_baseline], device=self.device)
            self.reward_buffer = []
    
        
        
    def train_step( self, batch: Dict[str, torch.Tensor], k: int = 10 ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Single training step"""
        post = batch['question']
        global count_warning, count_error
        paths = [p[0] for p in batch['topk_linearized_triplets']]
        answer = [p[0] for p in batch["answer"]]
        ques_embed = batch["question_embedding"].to(self.device)
        linearized_triplet_embeds = batch["topk_linearized_triplet_embeddings"].squeeze(0).to(self.device)
        relation_embeds = batch["topK_rel_embeddings"].squeeze(0).to(self.device)
        q_entity = [p[0] for p in batch['q_entity']]
        a_entity = [p[0] for p in batch['a_entity']]
        triplets = [(data[1][0][0], data[1][1][0], data[1][2][0]) for data in batch["topk_rel_data"]]
        graph_features = batch["graph_features"].squeeze(0).to(self.device)

        #print(ques_embed.shape,linearized_triplet_embeds.shape,relation_embeds.shape)
        if len(q_entity)==0:
            return None,None.to(self.device)
        ranking_scores, path_probs = self.path_ranker(ques_embed, linearized_triplet_embeds, relation_embeds, graph_features)
        #selected_paths, selected_probs, selected_ranking_scores, log_probs = self.path_ranker.sample_paths(path_probs, paths, k, ranking_scores)
        selected_triplets, selected_probs, selected_ranking_scores, log_probs = self.path_ranker.sample_paths(path_probs, triplets, k, ranking_scores)
        answer_reward = self.reward_func(selected_triplets, q_entity, a_entity)
        
        #answer_reward = compute_reward_v2(post, selected_triplets, q_entity, a_entity)
        if answer_reward is None:
            return None, None
        reward = torch.tensor([answer_reward], device=self.device)
        # Add to reward buffer for baseline update
        self.reward_buffer.append(reward.item())
        #path_importance = -lama_loss * selected_probs
        reinforcement_loss = self.compute_reinforce_loss(log_probs, reward.expand(log_probs.size(0)), torch.tensor([self.running_baseline], device=reward.device))
        # Total loss is LLM loss + REINFORCE loss
        #total_loss = reinforcement_loss
#         print("reward ",reward)
#         print("reinforcement_loss ", reinforcement_loss)
        return reinforcement_loss, reward
    
    @torch.no_grad()
    def validate(self, val_dataloader: DataLoader, k: int = 10) -> float:
        """Run validation loop"""
        self.path_ranker.eval()
        total_loss = 0
        total_ranking_loss = 0
        total_reward = 0
        valid_samples = 0

        for batch in tqdm(val_dataloader, desc="Validation"):
            loss, reward = self.train_step(batch, k)
            if loss is None:
                continue
            total_loss += loss.item()
            total_reward += reward.item()
            valid_samples += 1

        avg_loss = total_loss / valid_samples if valid_samples > 0 else 0
        avg_reward = total_reward / valid_samples if valid_samples > 0 else 0

        return avg_loss, avg_reward
    
    def save_checkpoint(self, epoch: int, val_loss: float, is_best: bool = False):
        print("saving model in epoch:{} and is best:{}".format(str(epoch),str(is_best)))
        if is_best:
            print("Best model with validation scores: ", val_loss)
            save_dir = os.path.join(self.checkpoint_dir, f"checkpoint_best_epoch_{epoch}")
        else:
            save_dir = os.path.join(self.checkpoint_dir, f"checkpoint_epoch_{epoch}")
        self.path_ranker.save_pretrained(save_dir)
        training_state = {
            'epoch': epoch,
            'val_loss': val_loss,
            'best_val_loss': self.best_val_loss
        }
        torch.save(training_state, os.path.join(save_dir, "training_state.pt"))
        print("Saved!!!")
    
    @classmethod
    def load_checkpoint(cls, checkpoint_dir: str, reward_func=None):
        """Load checkpoint using HuggingFace methods"""
        path_ranker = PathRankingModel.from_pretrained(checkpoint_dir)
        model = cls(path_ranker, reward_func)
        training_state = torch.load(os.path.join(checkpoint_dir, "training_state.pt"))
        model.best_val_loss = training_state['best_val_loss']
        
        print(training_state['epoch'], training_state['val_loss'])
        return model
    
    def train(
        self,
        monitor:TrainingMonitor,
        train_dataloader: DataLoader,
        val_dataloader: DataLoader,
        num_epochs: int = 3,
        k: int = 20,
        learning_rate: float = 1e-5,
        warmup_steps: int = 1000,
        scheduler_type: str = 'linear',
        validation_interval: int = 1,
        early_stopping_patience: int = 3,
    ):
        optimizer = torch.optim.AdamW([
            {'params': self.path_ranker.parameters(), 'lr': learning_rate}
        ])
        total_steps = (len(train_dataloader) * num_epochs) // self.gradient_accumulation_steps
        scheduler_func = get_cosine_schedule_with_warmup if scheduler_type == 'cosine' else get_linear_schedule_with_warmup
        scheduler = scheduler_func(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        self.best_val_loss = float("inf")
        #self.model.eval()
        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch + 1}/{num_epochs} - Training...")
            self.path_ranker.train()
            
            
            epoch_rewards = []
            epoch_losses = []
            epoch_reinforce_losses = []
            epoch_advantages = []  # Fixed variable name
            epoch_baselines = []   # Track baseline values too
            
            total_train_loss = 0
            optimizer.zero_grad()
            valid_batch_count =0
            
            with tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}", unit="batch") as pbar:
                for batch_idx, batch in enumerate(pbar):
                    loss, reward = self.train_step(batch, k)
                    if loss is None and reward is None:
                        continue
                    #print(reward)
                    if math.isnan(reward):
                        print("--------------------->", reward)
                    
                    # Get current baseline BEFORE updating it
                    current_baseline = self.path_ranker.baseline.detach().item() if hasattr(self.path_ranker, 'baseline') else self.running_baseline
                    advantage = reward.item() - current_baseline

                    # Store metrics
                    epoch_rewards.append(reward.item())
                    epoch_losses.append(loss.item())
                    epoch_reinforce_losses.append(loss.item())
                    epoch_advantages.append(advantage)
                    epoch_baselines.append(current_baseline)
                    
                    # Scale loss for gradient accumulation
                    valid_batch_count += 1
                    scaled_loss = loss / self.gradient_accumulation_steps
                    scaled_loss.backward()
                    
                    if valid_batch_count % self.gradient_accumulation_steps == 0 or (batch_idx + 1) == len(train_dataloader):
                        # Clip gradients before optimizer step
                        self.update_baseline_with_buffer()
                        torch.nn.utils.clip_grad_norm_(self.path_ranker.parameters(), self.max_grad_norm)
                        optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad()
                        
                        monitor.log_batch_metrics(
                            epoch * len(train_dataloader) + batch_idx, 
                            reward.item(), 
                            loss.item()
                        )
                        
                    total_train_loss += loss.item()
                    if (batch_idx + 1) % (self.gradient_accumulation_steps * 10) == 0:
                        avg_loss = total_train_loss / (batch_idx + 1)
                        current_lr = scheduler.get_last_lr()[0]
                        pbar.set_postfix({
                                'train_loss': f"{avg_loss:.2f}",
                                'lr': f"{current_lr:.2e}"
                            })
                
            train_metrics = {
                'loss': np.mean(epoch_losses),
                'reward': np.mean(epoch_rewards),
                'reinforce_loss': np.mean(epoch_reinforce_losses)
            }

            if (epoch + 1) % validation_interval == 0 and val_dataloader is not None:
                val_loss, val_reward= self.validate(val_dataloader, k)
                print(f"Epoch {epoch+1}/{num_epochs}, "
                      f"Validation Loss: {val_loss:.4f}, "
                      f"Validation reward: {val_reward:.4f}")
                
                val_metrics = {
                    'loss': val_loss,
                    'reward': val_reward,
                    'reinforce_loss': val_loss
                }
                monitor.log_epoch_metrics(epoch+1, train_metrics, val_metrics, optimizer, self.path_ranker)
                monitor.log_gradient_norm(self.path_ranker)


                if val_reward > self.best_val_reward:
                    self.best_val_reward = val_reward
                    patience_counter = 0
                    self.save_checkpoint(epoch + 1, val_loss, is_best=True)
                else:
                    self.save_checkpoint(epoch + 1, val_loss, is_best=False)
            avg_train_loss = total_train_loss / len(train_dataloader)
            print(f"Epoch {epoch+1}/{num_epochs}:")
            print(f"  Average Train Loss: {avg_train_loss:.4f}")
            print(f"  Average Reward: {np.mean(epoch_rewards):.4f}")
            print(f"  Average Advantage: {np.mean(epoch_advantages):.4f}")
            print(f"  Advantage Std: {np.std(epoch_advantages):.4f}")
            print(f"  Final Baseline: {self.running_baseline:.4f}")
            global reward_error, reward_error_list
            print("error count", reward_error)
            reward_error_list.append(reward_error)
            reward_error=0
            plt.figure(figsize=(15, 5))

            # Plot 1: Advantage distribution
            plt.subplot(1, 3, 1)
            plt.hist(epoch_advantages, bins=50, alpha=0.7, edgecolor='black')
            plt.axvline(np.mean(epoch_advantages), color='red', linestyle='--', 
                       label=f'Mean: {np.mean(epoch_advantages):.3f}')
            plt.axvline(0, color='black', linestyle='-', alpha=0.5, label='Zero')
            plt.title(f"Epoch {epoch+1} — Advantage Distribution")
            plt.xlabel("Advantage (Reward - Baseline)")
            plt.ylabel("Count")
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 2: Reward vs Baseline over time
            plt.subplot(1, 3, 2)
            batch_indices = range(len(epoch_rewards))
            plt.plot(batch_indices, epoch_rewards, alpha=0.6, label='Rewards')
            plt.plot(batch_indices, epoch_baselines, alpha=0.8, label='Baseline')
            plt.title(f"Epoch {epoch+1} — Rewards vs Baseline")
            plt.xlabel("Batch Index")
            plt.ylabel("Value")
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 3: Advantage over time
            plt.subplot(1, 3, 3)
            plt.plot(batch_indices, epoch_advantages, alpha=0.7, color='green')
            plt.axhline(0, color='black', linestyle='-', alpha=0.5)
            plt.title(f"Epoch {epoch+1} — Advantage Over Time")
            plt.xlabel("Batch Index")
            plt.ylabel("Advantage")
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

            # Additional statistics
            if len(epoch_advantages) > 0:
                print(f"  Advantage Range: [{np.min(epoch_advantages):.4f}, {np.max(epoch_advantages):.4f}]")
                print(f"  Positive Advantages: {np.sum(np.array(epoch_advantages) > 0)} / {len(epoch_advantages)}")

        monitor.plot_training_progress(save_plots=True, show_plots=True)
        monitor.plot_batch_level_analysis()
        monitor.save_metrics()
        monitor.print_training_summary()

In [ ]:
import os
import pickle

from transformers import (
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)


In [ ]:
class SampledJointTrainingDataset(Dataset):
    def __init__(self, dataset, k=500):
        self.precomputed_data = dataset
        self.k=k

    def __len__(self):
        return len(self.precomputed_data)

    def __getitem__(self, idx):
        data = self.precomputed_data[idx]
        if data["topk_linearized_triplet_embeddings"].shape[0] >=self.k:
            use_nums = self.k
        else:
            use_nums = data["topk_linearized_triplet_embeddings"].shape[0] 
        return {
            "question": data["question"],
            "is_empty": data["is_empty"],
            "q_entity": data["q_entity"],
            "a_entity": data["a_entity"],
            "answer": data["answer"],
            "question_embedding": data["question_embedding"], 
            "topk_linearized_triplets": data["topk_linearized_triplets"][:use_nums],
            "topk_linearized_triplet_embeddings": data["topk_linearized_triplet_embeddings"][:use_nums],
            "topk_rel_data": data["topk_rel_data"][:use_nums],
            "topK_rel_embeddings": data["topK_rel_embeddings"][:use_nums],
            "graph_features": data["graph_features"][:use_nums]
        }


In [ ]:
# topk_trn_dataset = SampledJointTrainingDataset(ppr_train_dataset)
# topk_val_dataset = SampledJointTrainingDataset(ppr_val_dataset)

# train_dataloader = DataLoader(topk_trn_dataset, batch_size=1, shuffle=True)
# val_dataloader = DataLoader(topk_val_dataset, batch_size=1, shuffle=False)

### REWARD V8

In [ ]:
# trained path ranker for n=1000 with pretrained n=500, with revard v8

topk_trn_dataset = SampledJointTrainingDataset(ptrain_dataset,k=1000)
topk_val_dataset = SampledJointTrainingDataset(pval_dataset,k=1000)

train_dataloader = DataLoader(topk_trn_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(topk_val_dataset, batch_size=1, shuffle=False)

path_ranker_model = torch.load("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/path_ranker_pretrained/architecture-v8/v1-n500_cosine/best_pretrained_model-5.pt", weights_only=False)
path_ranker1 = PathRankingModel()
path_ranker1.load_state_dict(path_ranker_model["model_state_dict"])

# for p in path_ranker1.parameters():
#     print(p)

base_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv8-n1000-e30_cosine"
monitor = TrainingMonitor(save_dir=os.path.join(base_dir,"training_logs"))
trainer = JointTrainer(
    path_ranker1,
    compute_reward_v8,
    max_grad_norm=1.0,
    gradient_accumulation_steps=32,
    checkpoint_dir=base_dir,
    gamma=0.99,  # discount factor
    baseline_decay=0.9  # baseline update rate
)

trainer.train(
    monitor,
    train_dataloader,
    val_dataloader,
    num_epochs=30,
    learning_rate=1e-4,
    warmup_steps=100,
    scheduler_type='cosine',
    validation_interval=1,  
    early_stopping_patience=3,
    k=100
)

In [ ]:
# trained path ranker for n=1000 with pretrained n=500, with revard v8

topk_trn_dataset = SampledJointTrainingDataset(ptrain_dataset,k=1000)
topk_val_dataset = SampledJointTrainingDataset(pval_dataset,k=1000)

train_dataloader = DataLoader(topk_trn_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(topk_val_dataset, batch_size=1, shuffle=False)

path_ranker_model = torch.load("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/path_ranker_pretrained/architecture-v8/v1-n500_cosine/best_pretrained_model-5.pt", weights_only=False)
path_ranker1 = PathRankingModel()
path_ranker1.load_state_dict(path_ranker_model["model_state_dict"])

# for p in path_ranker1.parameters():
#     print(p)

base_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv8-n1000-e30-k50_cosine"
monitor = TrainingMonitor(save_dir=os.path.join(base_dir,"training_logs"))
trainer = JointTrainer(
    path_ranker1,
    compute_reward_v8,
    max_grad_norm=1.0,
    gradient_accumulation_steps=32,
    checkpoint_dir=base_dir,
    gamma=0.99,  # discount factor
    baseline_decay=0.9  # baseline update rate
)

trainer.train(
    monitor,
    train_dataloader,
    val_dataloader,
    num_epochs=30,
    learning_rate=1e-4,
    warmup_steps=100,
    scheduler_type='cosine',
    validation_interval=1,  
    early_stopping_patience=3,
    k=50
)

In [ ]:
# trained path ranker for n=1000 with pretrained n=500, with revard v8

topk_trn_dataset = SampledJointTrainingDataset(ptrain_dataset,k=1000)
topk_val_dataset = SampledJointTrainingDataset(pval_dataset,k=1000)

train_dataloader = DataLoader(topk_trn_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(topk_val_dataset, batch_size=1, shuffle=False)

path_ranker_model = torch.load("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/path_ranker_pretrained/architecture-v8/v1-n500_cosine/best_pretrained_model-5.pt", weights_only=False)
path_ranker1 = PathRankingModel()
path_ranker1.load_state_dict(path_ranker_model["model_state_dict"])

# for p in path_ranker1.parameters():
#     print(p)

base_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv8-n1000-e30-k30_cosine"
monitor = TrainingMonitor(save_dir=os.path.join(base_dir,"training_logs"))
trainer = JointTrainer(
    path_ranker1,
    compute_reward_v8,
    max_grad_norm=1.0,
    gradient_accumulation_steps=32,
    checkpoint_dir=base_dir,
    gamma=0.99,  # discount factor
    baseline_decay=0.9  # baseline update rate
)

trainer.train(
    monitor,
    train_dataloader,
    val_dataloader,
    num_epochs=30,
    learning_rate=1e-4,
    warmup_steps=100,
    scheduler_type='cosine',
    validation_interval=1,  
    early_stopping_patience=3,
    k=30
)

## TRAINING MONITOR

In [ ]:
print(1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from collections import defaultdict
import json
import os

class TrainingMonitor:
    def __init__(self, save_dir="training_logs"):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        
        # Training metrics storage
        self.metrics = {
            'train_loss': [],
            'train_reward': [],
            'train_reinforce_loss': [],
            'val_loss': [],
            'val_reward': [],
            'val_reinforce_loss': [],
            'learning_rate': [],
            'baseline_value': [],
            'temperature': [],
            'selection_entropy': [],
            'gradient_norm': [],
            'epochs': []
        }
        
        # Detailed per-batch metrics (for smoothing)
        self.batch_metrics = {
            'batch_rewards': [],
            'batch_losses': [],
            'batch_numbers': []
        }
    
    def log_epoch_metrics(self, epoch, train_metrics, val_metrics, optimizer, model):
        """Log metrics for an epoch"""
        self.metrics['epochs'].append(epoch)
        
        # Training metrics
        self.metrics['train_loss'].append(train_metrics.get('loss', 0))
        self.metrics['train_reward'].append(train_metrics.get('reward', 0))
        self.metrics['train_reinforce_loss'].append(train_metrics.get('reinforce_loss', 0))
        
        # Validation metrics
        self.metrics['val_loss'].append(val_metrics.get('loss', 0))
        self.metrics['val_reward'].append(val_metrics.get('reward', 0))
        self.metrics['val_reinforce_loss'].append(val_metrics.get('reinforce_loss', 0))
        
        # Model metrics
        self.metrics['learning_rate'].append(optimizer.param_groups[0]['lr'])
        self.metrics['baseline_value'].append(model.baseline.item())
        self.metrics['temperature'].append(model.temperature.item())
        
        # Calculate selection entropy (measure of exploration)
        if hasattr(model, 'last_path_probs'):
            entropy = -torch.sum(model.last_path_probs * torch.log(model.last_path_probs + 1e-10))
            self.metrics['selection_entropy'].append(entropy.item())
        else:
            self.metrics['selection_entropy'].append(0)
    
    def log_batch_metrics(self, batch_num, reward, loss):
        """Log per-batch metrics for detailed analysis"""
        self.batch_metrics['batch_numbers'].append(batch_num)
        self.batch_metrics['batch_rewards'].append(reward)
        self.batch_metrics['batch_losses'].append(loss)
    
    def log_gradient_norm(self, model):
        """Log gradient norms for debugging"""
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** (1. / 2)
        self.metrics['gradient_norm'].append(total_norm)
    
    def plot_training_progress(self, save_plots=True, show_plots=True):
        """Generate comprehensive training progress plots""" 
        
        # Create figure with subplots
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Training Progress Monitoring', fontsize=16, fontweight='bold')
        
        epochs = self.metrics['epochs']
        
        # 1. Loss curves
        ax1 = axes[0, 0]
        if self.metrics['train_loss']:
            ax1.plot(epochs, self.metrics['train_loss'], 'b-', label='Train Loss', linewidth=2)
        if self.metrics['val_loss']:
            ax1.plot(epochs, self.metrics['val_loss'], 'r-', label='Val Loss', linewidth=2)
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training & Validation Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Reward curves (MOST IMPORTANT)
        ax2 = axes[0, 1]
        if self.metrics['train_reward']:
            ax2.plot(epochs, self.metrics['train_reward'], 'b-', label='Train Reward', linewidth=2)
        if self.metrics['val_reward']:
            ax2.plot(epochs, self.metrics['val_reward'], 'r-', label='Val Reward', linewidth=2)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Average Reward')
        ax2.set_title('Answer Coverage (Reward)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        #ax2.set_ylim(0, 1.0)  # Rewards are 0-1
        
        # 3. REINFORCE loss
        ax3 = axes[0, 2]
        if self.metrics['train_reinforce_loss']:
            ax3.plot(epochs, self.metrics['train_reinforce_loss'], 'g-', label='REINFORCE Loss', linewidth=2)
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('REINFORCE Loss')
        ax3.set_title('REINFORCE Loss')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 4. Learning rate and baseline
        ax4 = axes[1, 0]
        ax4_twin = ax4.twinx()
        
        if self.metrics['learning_rate']:
            line1 = ax4.plot(epochs, self.metrics['learning_rate'], 'purple', label='Learning Rate', linewidth=2)
        if self.metrics['baseline_value']:
            line2 = ax4_twin.plot(epochs, self.metrics['baseline_value'], 'orange', label='Baseline', linewidth=2)
        
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Learning Rate', color='purple')
        ax4_twin.set_ylabel('Baseline Value', color='orange')
        ax4.set_title('Learning Rate & Baseline')
        
        # Combine legends
        lines1, labels1 = ax4.get_legend_handles_labels()
        lines2, labels2 = ax4_twin.get_legend_handles_labels()
        ax4.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        ax4.grid(True, alpha=0.3)
        
        # 5. Temperature and entropy
        ax5 = axes[1, 1]
        ax5_twin = ax5.twinx()
        
        if self.metrics['temperature']:
            line1 = ax5.plot(epochs, self.metrics['temperature'], 'red', label='Temperature', linewidth=2)
        if self.metrics['selection_entropy']:
            line2 = ax5_twin.plot(epochs, self.metrics['selection_entropy'], 'blue', label='Selection Entropy', linewidth=2)
        
        ax5.set_xlabel('Epoch')
        ax5.set_ylabel('Temperature', color='red')
        ax5_twin.set_ylabel('Selection Entropy', color='blue')
        ax5.set_title('Temperature & Exploration')
        
        lines1, labels1 = ax5.get_legend_handles_labels()
        lines2, labels2 = ax5_twin.get_legend_handles_labels()
        ax5.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        ax5.grid(True, alpha=0.3)
        
        # 6. Gradient norms
        ax6 = axes[1, 2]
        if self.metrics['gradient_norm']:
            ax6.plot(epochs, self.metrics['gradient_norm'], 'brown', linewidth=2)
        ax6.set_xlabel('Epoch')
        ax6.set_ylabel('Gradient Norm')
        ax6.set_title('Gradient Magnitude')
        ax6.grid(True, alpha=0.3)
        ax6.set_yscale('log')  # Log scale for gradient norms
        
        plt.tight_layout()
        
        if save_plots:
            plt.savefig(f'{self.save_dir}/training_progress.png', dpi=300, bbox_inches='tight')
            print(f"Training plots saved to {self.save_dir}/training_progress.png")
        
        if show_plots:
            plt.show()
        else:
            plt.close()
    
    def plot_batch_level_analysis(self, window_size=100):
        """Plot detailed batch-level metrics with smoothing"""
        if not self.batch_metrics['batch_rewards']:
            print("No batch-level metrics to plot")
            return
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))
        
        batch_nums = self.batch_metrics['batch_numbers']
        rewards = self.batch_metrics['batch_rewards']
        losses = self.batch_metrics['batch_losses']
        
        # Smooth the metrics using moving average
        def moving_average(data, window):
            return np.convolve(data, np.ones(window)/window, mode='valid')
        
        if len(rewards) > window_size:
            smooth_rewards = moving_average(rewards, window_size)
            smooth_losses = moving_average(losses, window_size)
            smooth_batches = batch_nums[window_size-1:]
            
            ax1.plot(batch_nums, rewards, alpha=0.3, color='blue', label='Raw Rewards')
            ax1.plot(smooth_batches, smooth_rewards, color='blue', linewidth=2, label=f'Smoothed (window={window_size})')
        else:
            ax1.plot(batch_nums, rewards, color='blue', linewidth=2, label='Batch Rewards')
        
        ax1.set_xlabel('Batch Number')
        ax1.set_ylabel('Reward')
        ax1.set_title('Batch-Level Reward Progress')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(0, 1.0)
        
        # Loss plot
        if len(losses) > window_size:
            ax2.plot(batch_nums, losses, alpha=0.3, color='red', label='Raw Losses')
            ax2.plot(smooth_batches, smooth_losses, color='red', linewidth=2, label=f'Smoothed (window={window_size})')
        else:
            ax2.plot(batch_nums, losses, color='red', linewidth=2, label='Batch Losses')
        
        ax2.set_xlabel('Batch Number')
        ax2.set_ylabel('Loss')
        ax2.set_title('Batch-Level Loss Progress')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.save_dir}/batch_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def save_metrics(self):
        """Save metrics to JSON for later analysis"""
        with open(f'{self.save_dir}/training_metrics.json', 'w') as f:
            json.dump(self.metrics, f, indent=2)
        
        with open(f'{self.save_dir}/batch_metrics.json', 'w') as f:
            json.dump(self.batch_metrics, f, indent=2)
    
    def print_training_summary(self):
        """Print a summary of training progress"""
        if not self.metrics['epochs']:
            print("No training metrics available")
            return
        
        print("\n" + "="*60)
        print("TRAINING SUMMARY")
        print("="*60)
        
        last_epoch = self.metrics['epochs'][-1]
        print(f"Epochs Trained: {last_epoch}")
        
        if self.metrics['train_reward']:
            print(f"Final Train Reward: {self.metrics['train_reward'][-1]:.4f}")
            print(f"Best Train Reward: {max(self.metrics['train_reward']):.4f}")
        
        if self.metrics['val_reward']:
            print(f"Final Val Reward: {self.metrics['val_reward'][-1]:.4f}")
            print(f"Best Val Reward: {max(self.metrics['val_reward']):.4f}")
        
        if self.metrics['baseline_value']:
            print(f"Final Baseline: {self.metrics['baseline_value'][-1]:.4f}")
        
        if self.metrics['temperature']:
            print(f"Final Temperature: {self.metrics['temperature'][-1]:.4f}")
        
        # Check for potential issues
        print("\nTraining Health Check:")
        if self.metrics['val_reward'] and max(self.metrics['val_reward']) < 0.3:
            print("⚠️  Low validation rewards - consider longer training or check data")
        
        if self.metrics['gradient_norm'] and self.metrics['gradient_norm'][-1] < 1e-6:
            print("⚠️  Very small gradients - possible vanishing gradient problem")
        
        if self.metrics['gradient_norm'] and self.metrics['gradient_norm'][-1] > 10:
            print("⚠️  Large gradients - consider gradient clipping")
        
        print("="*60)


In [ ]:
print(101)

In [ ]:
monitor.metrics

## Test

In [ ]:
import torch


icl_user_prompt1 = """
Triplets:
- Lou Seal, sports.mascot.team, San Francisco Giants
- San Francisco Giants, sports.sports_team.championships, 2012 World Series
- San Francisco Giants, sports.sports_team.championships, 2010 World Series
- San Francisco Giants, sports.sports_team.championships, 2014 World Series
- Crazy Crab, sports.mascot.team, San Francisco Giants
- San Francisco Giants, sports.professional_sports_team.owner_s, Bill Gates
- New York Yankees, sports.sports_team.championships, 2009 World Series
- San Francisco Giants, sports.sports_team.colors, Blue and White
- m.0k079qm, base.schemastaging.team_training_ground_relationship.team, San Francisco Giants
- m.0k079ry, base.schemastaging.team_training_ground_relationship.team, San Francisco Giants

Question:
What year did the team with mascot named Lou Seal win the World Series?"""

icl_ass_prompt1  = """Answer in JSON format:
{"ans" : ["2014 (2014 World Series)", "2012 (2012 World Series)", "2010 (2010 World Series)"] }

Reason:
To answer the question, we need to:
1. Identify the team with the mascot Lou Seal.
2. Find the years that team won the World Series.

From the triplets, we can see that Lou Seal is the mascot of the San Francisco Giants.
Now, we need to find the year the San Francisco Giants won the World Series.
From the triplets, we can see that San Francisco Giants won the 2010 World Series and 2012 World Series and 2014 World Series.
So, the team with mascot named Lou Seal (San Francisco Giants) won the World Series in 2010, 2012, and 2014.

Therefore, the team with mascot Lou Seal won the World Series in 2010, 2012, and 2014.
"""

icl_user_prompt2 = """
Triplets: 
- Steve Bisciotti, sports.professional_sports_team.owner_s, Baltimore Ravens
- Steve Bisciotti, sports.sports_team_owner.teams_owned, Baltimore Ravens
- Steve Bisciotti, organization.organization_founder.organizations_founded, Allegis Group

Question:
Who is the coach of the team owned by Steve Bisciotti?"""

icl_ass_prompt2="""Answer in JSON format:
{"ans": ["answer not available"]}

Reason:
Based on the given knowledge triplets, the coach of the team owned by Steve Bisciotti is not explicitly mentioned. However, it can be inferred that the team owned by Steve Bisciotti is the Baltimore Ravens, a professional sports team. Therefore, additional knowledge about the current coach of the Baltimore Ravens can be used to answer the question.
"""

In [ ]:

# def create_prompt_icl(question, triplets, topk):
# #     triplet_text = "\n".join([f"- ({triplet[0]}, {triplet[1]}, {triplet[2]})" for triplet in triplets[:topk]])
#     triplet_text = "\n".join([f"- {triplet}" for triplet in triplets[:topk]])
#     na_format = '{\"ans\": [\"answer not available\"]}'
#     ans_format = '{\"ans\":[\"your answer 1\",\"your answer 2\"]}'
#     user_query = f"""
# Linearized Triplets: 
# {triplet_text}

# Question:
# {question}

# Let's think step by step."""
    
#     prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
# You are a knowledge graph question answering system. Given a question and relevant linearized knowledge triplets, extract all the correct answers supported by the triplets in JSON format and provide a brief reason for the answer based on the triplets.

# Instructions:
# 1> Provide your final answer in JSON format using this placeholder: {ans_format}. If there is insufficient information to answer the question, return {na_format}.
# 2> Avoid duplicate entries in the answer JSON.
# 3> Keep your reasoning brief and focused.
# 4> Your answer must not contradict any information presented in the provided triplets.
# 5> If the answer is directly supported by the triplets, use the triplets to justify your answer.
# 6> If the answer is not explicitly found in the triplets, you may use your own factual knowledgebut only if it is consistent with the information in the triplets.

# #Example 1:
# {icl_user_prompt1}
# #Answer:
# {icl_ass_prompt1}

# #Example 2:
# {icl_user_prompt2}
# #Answer:
# {icl_ass_prompt2}

# Now consider the below Triplets and answer the Question carefully
# <|start_header_id|>user<|end_header_id|>\n
# {user_query}
# #Answer:
# <|start_header_id|>assistant<|end_header_id|>
#     """
#     return prompt


def create_prompt_icl(question, triplets, topk):
#     triplet_text = "\n".join([f"- ({triplet[0]}, {triplet[1]}, {triplet[2]})" for triplet in triplets[:topk]])
    triplet_text = "\n".join([f"- {triplet}" for triplet in triplets[:topk]])
    na_format = '{\"ans\": [\"answer not available\"]}'
    ans_format = '{\"ans\":[\"your answer 1\",\"your answer 2\"]}'
    user_query = f"""
Linearized Triplets: 
{triplet_text}

Question:
{question}

Let's think step by step."""
    
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a knowledge graph question answering system. Given a question and relevant linearized knowledge triplets, provide correct answers in JSON format supported by the triplets and provide a brief reason for the answer based on the triplets.

Instructions:
1> Provide your final answer in JSON format using this placeholder: {ans_format}. If there is insufficient information to answer the question, return {na_format}.
2> Ensure answer does not contain duplicate entries.
3> Keep your reasoning brief and focused.
4> Your answer must not contradict any information presented in the provided triplets.
5> If the answer is directly supported by the triplets, use the triplets to justify your answer.
6> If the answer is not explicitly found in the triplets, you may use your own factual knowledgebut only if it is consistent with the information in the triplets.

#Example 1:
{icl_user_prompt1}
#Answer:
{icl_ass_prompt1}

#Example 2:
{icl_user_prompt2}
#Answer:
{icl_ass_prompt2}

Now consider the below Triplets and answer the Question carefully
<|start_header_id|>user<|end_header_id|>\n
{user_query}
#Answer:
<|start_header_id|>assistant<|end_header_id|>
    """
    return prompt
    

In [ ]:
def create_prompt_icl_openai(question, triplets, topk):
    triplet_text = "\n".join([f"- {triplet}" for triplet in triplets[:topk]])
    na_format = '{\"ans\": [\"answer not available\"]}'
    ans_format = '{\"ans\":[\"your answer 1\",\"your answer 2\"]}'

    # ICL Example 1
    icl_example_1 = {
        "role": "user",
        "content": icl_user_prompt1.strip()
    }
    icl_example_1_response = {
        "role": "assistant",
        "content": icl_ass_prompt1.strip()
    }

    # ICL Example 2
    icl_example_2 = {
        "role": "user",
        "content": icl_user_prompt2.strip()
    }
    icl_example_2_response = {
        "role": "assistant",
        "content": icl_ass_prompt2.strip()
    }
    
    # Test instance
    test_query = f"""
Linearized Triplets:
{triplet_text}

Question:
{question}

Let's think step by step.
""".strip()

    test_user = {
        "role": "user",
        "content": test_query
    }

    messages = [
        {
            "role": "system",
            "content": f"""You are a knowledge graph question answering system. Given a question and relevant knowledge triplets (linearized using space character), provide most possible answers in JSON format supported by the triplets and provide a brief reason for the answer based on the triplets.
Instructions:
1> Put the most confident answer in the at beginning in JSON response.
2> Provide your final answer in JSON format using this placeholder: {ans_format}. If there is insufficient information to answer the question, return {na_format}.
3> Keep your reasoning brief and focused.
4> Your answer must not contradict any information presented in the provided triplets.
5> If the answer is directly supported by the triplets, use the triplets to justify your answer.
6> If the answer is not explicitly found in the triplets, you may use your own factual knowledge but only if it is consistent with the information in the triplets.
"""
        },
        icl_example_1,
        icl_example_1_response,
        test_user
    ]

    return messages


In [ ]:
"""
Script to evaluate Llama 3.1-8B on KGQA using linearized triplets sorted by cosine similarity.
Computes Macro-F1, Hit@1, Precision, Recall, and Exact Match metrics.
"""


import openai
api_key = ""

client = openai.OpenAI(api_key=api_key)



json_match_error=[]
json_decode_error=[]
data_error=[]

def normalize(s: str) -> str:
    """Lower text and remove punctuation, articles and extra whitespace."""
    s = s.lower()
    exclude = set(string.punctuation)
    s = "".join(char for char in s if char not in exclude)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"\b(<pad>)\b", " ", s)
    s = " ".join(s.split())
    return s


def match(s1: str, s2: str) -> bool:
    """Check if s2 is contained in s1 after normalization."""
    s1 = normalize(s1)
    s2 = normalize(s2)
    return s2 in s1


def remove_duplicates(input_list):
    """Remove duplicates while preserving order."""
    seen = set()
    result = []
    for item in input_list:
        if item not in seen:
            result.append(item)
            seen.add(item)
    return result


def get_pred(prediction):
    """Extract predictions from model output."""
    response_error=0
    decode_error=0
    global json_match_error, json_decode_error, data_error
    pattern = r'\{[^{}]*\}'
    json_match = re.search(pattern, prediction)
    if not json_match:
        print("###############################>>No JSON match")
        if '{"ans": [' in prediction:
            x = prediction.split("""{"ans": [""")[-1]
            entities = [ele.strip().strip('"').lower() for ele in x.split(",")]
            response = remove_duplicates(entities)
            print("NJM response: ", response)
        else: 
                response = [prediction.strip()]
                print("NJM response no ans found: ", response)
                json_match_error.append({"response":prediction})
                response_error+=1
    else:
        json_str = json_match.group(0)
        try:
            data = json.loads(json_str)
            response = remove_duplicates(data["ans"])
            #print(response)
        except Exception as e:
            print("###############################>>Error decoding JSON")
            json_decode_error.append({"response":prediction})
            decode_error+=1
            response = [prediction.strip()]
            print("JDE response: ", response)
    if response_error>0 or decode_error>0:
        data_error.append((response_error,decode_error))
    return response


def eval_recall(prediction, answer, double_check):
    """Calculate recall score."""
    prediction = deepcopy(prediction)
    prediction = sorted(prediction, key=len, reverse=True)
    matched = 0.
    for a in answer:
        for pred in prediction:
            if match(pred, a):
                matched += 1
                prediction.remove(pred)
                break
            elif double_check:
                if match(a, pred.split('ans:')[-1].strip()) or match(a, pred):
                    matched += 1
                    prediction.remove(pred)
                    break
    return matched / len(answer), matched, len(answer)


def eval_precision(prediction, answer, double_check):
    """Calculate precision score."""
    prediction = deepcopy(prediction)
    prediction = sorted(prediction, key=len, reverse=True)
    num_pred = len(prediction)
    if num_pred == 0:
        return 0, 0, 0
    matched = 0.
    for a in answer:
        for pred in prediction:
            if match(pred, a):
                matched += 1
                prediction.remove(pred)
                break
            elif double_check:
                if match(a, pred.split('ans:')[-1].strip()) or match(a, pred):
                    matched += 1
                    prediction.remove(pred)
                    break
    return matched / num_pred, matched, num_pred


def eval_f1(precision, recall):
    """Calculate F1 score."""
    if precision + recall == 0:
        return 0
    return 2 * precision * recall / (precision + recall)


def eval_hit1(prediction, answer, double_check):
    """Calculate Hit@1 score."""
    if len(prediction) == 0:
        return 0
    for a in answer:
        if match(prediction[0], a):
            return 1
        elif double_check:
            if match(a, prediction[0].strip()):
                return 1
    return 0

def eval_hit(prediction, answer, double_check):
    """Calculate Hit score."""
    if len(prediction) == 0:
        return 0
    for a in answer:
        for p in prediction:
            if match(p, a):
                return 1
            elif double_check and match(a,p.strip()):
                return 1
    return 0

# def generate_answer(model, tokenizer, prompt, max_new_tokens=1024):
#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
#     inputs = {k: v.to(model.device) for k, v in inputs.items()}
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             temperature=0.1,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id,
#             eos_token_id=tokenizer.eos_token_id
#         )
#     #print(inputs['input_ids'].shape[1])
#     #pl=len(tokenizer.encode(prompt))
#     #print(pl)
#     response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
#     return response.strip()


def generate_answer(messages, max_new_tokens=1024):
    global client
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.1,
        max_completion_tokens=max_new_tokens
    )
    return response.choices[0].message.content.strip()


@torch.no_grad
def evaluate_dataset(tst_dataloader, output_dir, jointtrainer, top_k):
    os.makedirs(output_dir, exist_ok=True)
    global model, tokenizer
    predictions_file = os.path.join(output_dir, 'predictions.jsonl')
    detailed_results_file = os.path.join(output_dir, 'detailed_results.jsonl')
    hit_list = []
    hit1_list = []
    f1_list = []
    precision_list = []
    recall_list = []
    total_pred = 0
    total_answer = 0
    total_match = 0
    jointtrainer.path_ranker.eval()
    with open(predictions_file, 'w', encoding='utf-8') as pred_f, \
         open(detailed_results_file, 'w', encoding='utf-8') as detail_f:
        
        for i, batch in enumerate(tqdm(tst_dataloader)):
            try:
                question = batch['question'][0]
                paths = [p[0] for p in batch['topk_linearized_triplets']]
                ground_truth = [p[0] for p in batch["answer"]]
                if ground_truth == [] or len(ground_truth)==0:
                    continue
                ques_embed = batch["question_embedding"].to("cuda")
                linearized_triplet_embeds = batch["topk_linearized_triplet_embeddings"].squeeze(0).to("cuda")
                relation_embeds = batch["topK_rel_embeddings"].squeeze(0).to("cuda")
                graph_features = batch["graph_features"].squeeze(0).to("cuda")
                ranking_scores, path_probs = jointtrainer.path_ranker(ques_embed, linearized_triplet_embeds, relation_embeds, graph_features)
                selected_paths, selected_probs, selected_ranking_scores, log_probs = jointtrainer.path_ranker.sample_paths(path_probs, paths, top_k, ranking_scores)
                
                #Sort the paths based on probablities:
                sorted_indices = torch.argsort(selected_probs, descending=True)

                # Sort the paths accordingly
                sorted_paths = [selected_paths[i] for i in sorted_indices.tolist()]

                #prompt = create_prompt_icl(question, sorted_paths, top_k)
                prompt = create_prompt_icl_openai(question, sorted_paths, top_k)
                
                #raw_prediction = generate_answer(model, tokenizer, prompt)
                raw_prediction = generate_answer(prompt)
                
                #print(prompt)
                prediction_r = get_pred(raw_prediction)
                prediction = [s for s in prediction_r if s != ""]
                # Handle date questions
                answer = sorted(remove_duplicates(ground_truth), key=len, reverse=True)
                if 'when' in question.lower() or 'what year' in question.lower():
                    for idx in range(len(answer)):
                        if '-' in answer[idx] and answer[idx].split('-')[0].isdigit():
                            answer[idx] = answer[idx].split('-')[0]
                
                # Determine if double check is needed
                double_check = any([keyword in question.lower() for keyword in 
                                  ['when', 'what year', 'which year', 'where', 'sport', 
                                   "what countr", "language", 'nba finals', 'world series']])
                
                # Calculate metrics
                precision_score, matched_1, num_pred = eval_precision(prediction, answer, double_check)
                recall_score, matched_2, num_answer = eval_recall(prediction, answer, double_check)
                f1_score = eval_f1(precision_score, recall_score)
                hit1 = eval_hit1(prediction, answer, double_check)
                hit = eval_hit(prediction, answer, double_check)
                
                assert matched_1 == matched_2
                total_pred += num_pred
                total_answer += num_answer
                total_match += matched_1
                
                hit1_list.append(hit1)
                hit_list.append(hit)
                f1_list.append(f1_score)
                precision_list.append(precision_score)
                recall_list.append(recall_score)
                
                pred_data = {
                    'id': i,
                    'question': question,
                    'prediction': raw_prediction,
                    'processed_prediction': prediction,
                    'ground_truth': answer,
                }
                pred_f.write(json.dumps(pred_data, ensure_ascii=False) + '\n')
                
                detail_data = {
                    'id': i,
                    'question': question,
                    'prediction': prediction,
                    'ground_truth': answer,
                    'hit@1': hit1,
                    'hit':hit,
                    'f1': f1_score,
                    'precision': precision_score,
                    'recall': recall_score,
                }
                detail_f.write(json.dumps(detail_data, ensure_ascii=False) + '\n')
                
            except Exception as e:
                print(f"Error processing item {i}: {e}")
                continue
    
    if len(hit_list) == 0:
        print("No valid predictions found!")
        return
    
    avg_hit = sum(hit_list) * 100 / len(hit_list)
    avg_hit1 = sum(hit1_list) * 100 / len(hit1_list)
    avg_f1 = sum(f1_list) * 100 / len(f1_list)
    avg_precision = sum(precision_list) * 100 / len(precision_list)
    avg_recall = sum(recall_list) * 100 / len(recall_list)
    
    num_exact_match = (np.array(f1_list) == 1).sum() / len(f1_list) * 100
    num_totally_wrong = (np.array(recall_list) == 0).sum() / len(recall_list) * 100
    
    micro_precision = total_match / total_pred if total_pred > 0 else 0
    micro_recall = total_match / total_answer if total_answer > 0 else 0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0
    
    # Print results
    result_str = f"Hit: {avg_hit:.2f}, Hit@1: {avg_hit1:.2f}, Macro F1: {avg_f1:.2f}, Macro Precision: {avg_precision:.2f}, Macro Recall: {avg_recall:.2f}, Exact Match: {num_exact_match:.2f}, Totally Wrong: {num_totally_wrong:.2f}"
    print(result_str)
    print(f"Micro F1: {micro_f1:.4f}, Micro Precision: {micro_precision:.4f}, Micro Recall: {micro_recall:.4f}")
    print(f"Total samples: {len(hit_list)}")
    
    # Save final results
    results_file = os.path.join(output_dir, 'final_results.txt')
    with open(results_file, 'w') as f:
        f.write(result_str + '\n')
        f.write(f"Micro F1: {micro_f1:.4f}, Micro Precision: {micro_precision:.4f}, Micro Recall: {micro_recall:.4f}\n")
        f.write(f"Total samples: {len(hit_list)}\n")
    
    return {
        'hit':avg_hit,
        'hit@1': avg_hit1,
        'macro_f1': avg_f1,
        'macro_precision': avg_precision,
        'macro_recall': avg_recall,
        'exact_match': num_exact_match,
        'totally_wrong': num_totally_wrong,
        'micro_f1': micro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall
    }

@torch.no_grad
def compare_triplets_generated(tst_dataloader, output_dir, jointtrainer, top_k):
    os.makedirs(output_dir, exist_ok=True)
    detailed_results_file = os.path.join(output_dir, 'example_selections.jsonl')
    jointtrainer.path_ranker.eval()
    with open(detailed_results_file, 'w', encoding='utf-8') as detail_f:
        for i, batch in enumerate(tqdm(tst_dataloader)):
            try:
                question = batch['question'][0]
                paths = [p[0] for p in batch['topk_linearized_triplets']]
                ground_truth = [p[0] for p in batch["answer"]]
                if ground_truth == [] or len(ground_truth)==0:
                    continue
                ques_embed = batch["question_embedding"].to("cuda")
                linearized_triplet_embeds = batch["topk_linearized_triplet_embeddings"].squeeze(0).to("cuda")
                relation_embeds = batch["topK_rel_embeddings"].squeeze(0).to("cuda")
                graph_features = batch["graph_features"].squeeze(0).to("cuda")
                ranking_scores, path_probs = jointtrainer.path_ranker(ques_embed, linearized_triplet_embeds, relation_embeds, graph_features)
                selected_paths, selected_probs, selected_ranking_scores, log_probs = jointtrainer.path_ranker.sample_paths(path_probs, paths, top_k, ranking_scores)
                sorted_indices = torch.argsort(selected_probs, descending=True)
                # Sort the paths accordingly
                sorted_paths = [selected_paths[i] for i in sorted_indices.tolist()]
                detail_data = {
                    "question":question,
                    "ground_truth":ground_truth,
                    "reranker":sorted_paths[:top_k],
                    "cosine":paths[:top_k]
                }
                detail_f.write(json.dumps(detail_data, ensure_ascii=False) + '\n')
                print("============================================================")
                print("-----------------> question: ", question)
                print("-----------------> answer: ", ground_truth)
                print("model selected paths:", sorted_paths[:20])
                print("*****************")
                print("cosine similarity",paths[:20])
                if i==50:
                    break
            except Exception as e:
                print(e)
                
    
@torch.no_grad
def generate_selected_json(tst_dataloader, output_dir, jointtrainer, top_k):
    os.makedirs(output_dir, exist_ok=True)
    detailed_results_file = os.path.join(output_dir, 'selected_triplets.json')
    results = []
    jointtrainer.path_ranker.eval()
    count=0
    for i, batch in enumerate(tqdm(tst_dataloader)):
            question = batch['question'][0]
            paths = [p[0] for p in batch['topk_linearized_triplets']]
            ground_truth = [p[0] for p in batch["answer"]]
            if ground_truth == [] or len(ground_truth)==0 or len(paths)==0:
                continue
            if len(paths) >= top_k:
                ques_embed = batch["question_embedding"].to("cuda")
                linearized_triplet_embeds = batch["topk_linearized_triplet_embeddings"].squeeze(0).to("cuda")
                relation_embeds = batch["topK_rel_embeddings"].squeeze(0).to("cuda").to("cuda")
                graph_features = batch["graph_features"].squeeze(0).to("cuda")
                #print(ques_embed.shape, linearized_triplet_embeds.shape, relation_embeds.shape, graph_features.shape)
                ranking_scores, path_probs = jointtrainer.path_ranker(ques_embed, linearized_triplet_embeds, relation_embeds, graph_features)
                selected_paths, selected_probs, selected_ranking_scores, log_probs = jointtrainer.path_ranker.sample_paths(path_probs, paths, top_k, ranking_scores)
                sorted_indices = torch.argsort(selected_probs, descending=True)
                sorted_paths = [selected_paths[i] for i in sorted_indices.tolist()]
                detail_data = {
                    "question":question,
                    "answer":ground_truth,
                    "a_entity": [p[0] for p in batch["a_entity"]],
                    "reranker":sorted_paths,
                }
            else:
                detail_data = {
                    "question":question,
                    "answer":ground_truth,
                    "a_entity": [p[0] for p in batch["a_entity"]],
                    "reranker":paths,
                }
                count+=1
            results.append(detail_data)
    with open(detailed_results_file, "w") as f:
        json.dump(results,f)
    return count
    

In [ ]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
for param in model.parameters():
    param.requires_grad = False  # Freezing the model

In [ ]:
# joint_trainer = JointTrainer.load_checkpoint("/mnt/LS226/LS25/sourav23099/cwq/cwq-v21/model/architecture-v7/scorer-gate-ppr-v2-modified/checkpoint_epoch_10")

In [ ]:
class SampledJointTrainingDataset(Dataset):
    def __init__(self, dataset,k):
        self.dataset = dataset
        self.k=k

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        data = self.dataset[idx]
        if data["topk_linearized_triplet_embeddings"].shape[0] <= self.k:
            use_k = data["topk_linearized_triplet_embeddings"].shape[0]
        else:
            use_k = self.k
        return {
            "question": data["question"],
            "is_empty": data["is_empty"],
            "q_entity": data["q_entity"],
            "a_entity": data["a_entity"],
            "answer": data["answer"],
            "question_embedding": data["question_embedding"],
            "topk_linearized_triplets": data["topk_linearized_triplets"][:use_k],
            "topk_linearized_triplet_embeddings": data["topk_linearized_triplet_embeddings"][:use_k],
            "topk_rel_data": data["topk_rel_data"][:use_k],
            "topK_rel_embeddings": data["topK_rel_embeddings"][:use_k],
            "graph_features": data["graph_features"][:use_k]
        }
    
test_dataset = torch.load("/mnt/LS226/LS25/sourav23099/cwq/cwq-v21/test/test_jointrainer_path_dataset_v3_ppr.pt", weights_only=False)

In [ ]:
filtered_tst_dataset = SampledJointTrainingDataset(test_dataset,k=1000)
print("test dataset created")
print("Len of triplets:", len(filtered_tst_dataset[1]["topk_linearized_triplets"]))
tst_dataloader = DataLoader(filtered_tst_dataset, batch_size=1, shuffle=False)
output_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/results/architecture-v8/v7-rv4-n1000-e50_ablation1"
generate_selected_json(tst_dataloader,
    output_dir,
    trainer,
    top_k=100)
compare_triplets_generated(tst_dataloader,
    output_dir,
    trainer,
    top_k=100)

In [ ]:
filtered_tst_dataset = SampledJointTrainingDataset(test_dataset,k=1000)
print("test dataset created")
print("Len of triplets:", len(filtered_tst_dataset[1]["topk_linearized_triplets"]))
tst_dataloader = DataLoader(filtered_tst_dataset, batch_size=1, shuffle=False)
output_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/results/architecture-v8/v7-rv4-n1000-e50_ablation2"
generate_selected_json(tst_dataloader,
    output_dir,
    trainer,
    top_k=100)
compare_triplets_generated(tst_dataloader,
    output_dir,
    trainer,
    top_k=100)

In [ ]:
#Reward v8

In [ ]:
joint_trainer_best = JointTrainer.load_checkpoint("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv8-n1000-e30_cosine/checkpoint_best_epoch_30")
filtered_tst_dataset = SampledJointTrainingDataset(test_dataset,k=1000)
print("test dataset created")
print("Len of triplets:", len(filtered_tst_dataset[1]["topk_linearized_triplets"]))
tst_dataloader = DataLoader(filtered_tst_dataset, batch_size=1, shuffle=False)
output_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/results/architecture-v8/v7-rv8-n1000-e30_cosine"
generate_selected_json(tst_dataloader,
    output_dir,
    joint_trainer_best,
    top_k=100)
compare_triplets_generated(tst_dataloader,
    output_dir,
    joint_trainer_best,
    top_k=100)

In [ ]:
filtered_tst_dataset = SampledJointTrainingDataset(test_dataset,k=1000)
print("test dataset created")
print("Len of triplets:", len(filtered_tst_dataset[1]["topk_linearized_triplets"]))
tst_dataloader = DataLoader(filtered_tst_dataset, batch_size=1, shuffle=False)
output_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/results/architecture-v8/v7-rv4-n1000-e50_ablation2"
generate_selected_json(tst_dataloader,
    output_dir,
    trainer,
    top_k=100)
compare_triplets_generated(tst_dataloader,
    output_dir,
    trainer,
    top_k=100)

## Ablation

In [ ]:
def triplet_ablation(tst_dataloader, output_dir, jointtrainer, top_k, filename='triplet_ablations.jsonl'):
    import os
    import json
    from tqdm import tqdm
    import torch

    os.makedirs(output_dir, exist_ok=True)
    detailed_results_file = os.path.join(output_dir,filename )
    jointtrainer.path_ranker.eval()

    total_common = 0
    total_diff = 0
    count = 0

    with open(detailed_results_file, 'w', encoding='utf-8') as detail_f:
        for i, batch in enumerate(tqdm(tst_dataloader)):
            try:
                question = batch['question'][0]
                paths = [p[0] for p in batch['topk_linearized_triplets']]
                ground_truth = [p[0] for p in batch["answer"]]
                if len(paths)==0 or not ground_truth:
                    continue

                # Move tensors to GPU
                ques_embed = batch["question_embedding"].to("cuda")
                linearized_triplet_embeds = batch["topk_linearized_triplet_embeddings"].squeeze(0).to("cuda")
                relation_embeds = batch["topK_rel_embeddings"].squeeze(0).to("cuda")
                graph_features = batch["graph_features"].squeeze(0).to("cuda")

                # Get scores and sampled paths
                ranking_scores, path_probs = jointtrainer.path_ranker(
                    ques_embed, linearized_triplet_embeds, relation_embeds, graph_features
                )
                selected_paths, selected_probs, selected_ranking_scores, log_probs = jointtrainer.path_ranker.sample_paths(
                    path_probs, paths, top_k, ranking_scores
                )

                # Sort sampled paths by probability
                sorted_indices = torch.argsort(selected_probs, descending=True)
                sorted_paths = [selected_paths[i] for i in sorted_indices.tolist()]

                # Compare with top-K cosine paths
                top_cosine_paths = set(paths[:top_k])
                top_reranked_paths = set(sorted_paths[:top_k])

                common_paths = top_reranked_paths.intersection(top_cosine_paths)
                different_paths = top_reranked_paths.difference(top_cosine_paths)

                # Track global stats
                total_common += len(common_paths)
                total_diff += len(different_paths)
                count += 1

                # Prepare JSON output
                detail_data = {
                    "id": i,
                    "question": question,
                    "ground_truth": ground_truth,
                    "reranker": sorted_paths[:top_k],
                    "cosine": paths[:top_k],
                    "num_common_paths": len(common_paths),
                    "num_diff_paths": len(different_paths),
                    "distinct_model_paths": list(different_paths)
                }

                detail_f.write(json.dumps(detail_data, ensure_ascii=False) + '\n')

            except Exception as e:
                print(f"Error processing batch {i}: {e}")
                continue

    # Print global statistics
    if count > 0:
        print("\n========== Overall Statistics ==========")
        print(f"Average number of common paths:   {total_common / count:.2f}")
        print(f"Average number of distinct paths: {total_diff / count:.2f}")
    else:
        print("No valid examples processed.")


In [ ]:
test_dataset = torch.load("/mnt/LS226/LS25/sourav23099/cwq/cwq-v21/test/test_jointrainer_path_dataset_v3_ppr.pt", weights_only=False)
joint_trainer_gpt_e50 = JointTrainer.load_checkpoint("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv4-n1000_e50_cosine/checkpoint_epoch_50")
filtered_tst_dataset = SampledJointTrainingDataset(test_dataset,k=1000)
print("Len of triplets:", len(filtered_tst_dataset[1]["topk_linearized_triplets"]))
tst_dataloader = DataLoader(filtered_tst_dataset, batch_size=1, shuffle=False)
output_dir = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/results/architecture-v8/v7-rv4-n1000_e50_cosine_gpt"
triplet_ablation(tst_dataloader,
    output_dir,
    joint_trainer_gpt_e50,
    top_k=100)

In [ ]:
triplet_ablation(tst_dataloader,
    output_dir,
    joint_trainer_gpt_e50,
    top_k=50)

In [ ]:
import random
from torch.utils.data import DataLoader, Subset

# Fix the seed if you want reproducibility
random.seed(42)

# Randomly sample 100 unique indices
sampled_indices = random.sample(range(len(filtered_tst_dataset)), 100)

# Create a subset using these indices
sampled_dataset = Subset(filtered_tst_dataset, sampled_indices)

# Create DataLoader from the sampled dataset
sampled_dataloader = DataLoader(sampled_dataset, batch_size=1, shuffle=False)

In [ ]:
triplet_ablation(sampled_dataloader,
    output_dir,
    joint_trainer_gpt_e50,
    top_k=100,
    filename="tripelt_ablation-rand-k100.jsonl")

In [ ]:
triplet_ablation(sampled_dataloader,
    output_dir,
    joint_trainer_gpt_e50,
    top_k=50,
    filename="tripelt_ablation-rand-k50.jsonl")

In [ ]:
import os
import json
import torch
import networkx as nx
from tqdm import tqdm

def evaluate_answer_and_path_coverage(tst_dataloader, jointtrainer, top_k):
    jointtrainer.path_ranker.eval()

    count = 0

    cosine_ans_present = 0
    cosine_reasoning_path = 0

    model_ans_present = 0
    model_reasoning_path = 0

    for i, batch in enumerate(tqdm(tst_dataloader)):
        try:
            triplets = [(data[1][0][0], data[1][1][0], data[1][2][0]) for data in batch["topk_rel_data"]]
            if len(triplets) == 0:
                continue

            ques_ents = [p[0].lower() for p in batch["q_entity"]]
            ans_ents = [p[0].lower() for p in batch["a_entity"]]

            # ================= COSINE-SORTED TRIPLETS ====================
            top_cosine_triplets = triplets[:top_k]
            G_cosine = nx.Graph()  # Use undirected graph
            for s, p, o in top_cosine_triplets:
                G_cosine.add_edge(s.lower(), o.lower(), relation=p.lower())

            # Check answer entity presence in cosine triplets
            ans_present_cosine = any(
                ent in {s.lower(), o.lower()} for ent in ans_ents for (s, _, o) in top_cosine_triplets
            )
            if ans_present_cosine:
                cosine_ans_present += 1

            # Check reasoning path in cosine graph
            path_found_cosine = False
            for q_ent in ques_ents:
                for a_ent in ans_ents:
                    if q_ent in G_cosine.nodes and a_ent in G_cosine.nodes:
                        if nx.has_path(G_cosine, q_ent, a_ent):
                            path_found_cosine = True
                            break
                if path_found_cosine:
                    break
            if path_found_cosine:
                cosine_reasoning_path += 1

            # ================= MODEL-SELECTED TRIPLETS ====================
            ques_embed = batch["question_embedding"].to("cuda")
            triplet_embeds = batch["topk_linearized_triplet_embeddings"].squeeze(0).to("cuda")
            relation_embeds = batch["topK_rel_embeddings"].squeeze(0).to("cuda")
            graph_features = batch["graph_features"].squeeze(0).to("cuda")

            ranking_scores, path_probs = jointtrainer.path_ranker(
                ques_embed, triplet_embeds, relation_embeds, graph_features
            )

            selected_triplets, selected_probs, selected_ranking_scores, log_probs = jointtrainer.path_ranker.sample_paths(
                path_probs, triplets, top_k, ranking_scores
            )

            G_model = nx.Graph()  # Use undirected graph
            for s, p, o in selected_triplets:
                G_model.add_edge(s.lower(), o.lower(), relation=p.lower())

            # Check answer entity presence in model triplets
            ans_present_model = any(
                ent in {s.lower(), o.lower()} for ent in ans_ents for (s, _, o) in selected_triplets
            )
            if ans_present_model:
                model_ans_present += 1

            # Check reasoning path in model graph
            path_found_model = False
            for q_ent in ques_ents:
                for a_ent in ans_ents:
                    if q_ent in G_model.nodes and a_ent in G_model.nodes:
                        if nx.has_path(G_model, q_ent, a_ent):
                            path_found_model = True
                            break
                if path_found_model:
                    break
            if path_found_model:
                model_reasoning_path += 1
                
            count += 1
            
        except Exception as e:
            #print(f"Error in batch {i}: {e}")
            continue

    # ======= SUMMARY ========
    if count > 0:
        print("\n========== Reasoning & Answer Coverage Statistics ==========")
        print(f"Total evaluated samples: {count}")
        print(f"\n--- Cosine Similarity Top-{top_k} ---")
        print(f"Answer entity present:     {cosine_ans_present/count*100:.2f}% ({cosine_ans_present}/{count})")
        print(f"Reasoning path exists:     {cosine_reasoning_path/count*100:.2f}% ({cosine_reasoning_path}/{count})")

        print(f"\n--- Model Selected Top-{top_k} ---")
        print(f"Answer entity present:     {model_ans_present/count*100:.2f}% ({model_ans_present}/{count})")
        print(f"Reasoning path exists:     {model_reasoning_path/count*100:.2f}% ({model_reasoning_path}/{count})")
    else:
        print("No valid samples were processed.")


In [ ]:
filtered_tst_dataset = SampledJointTrainingDataset(test_dataset,k=1000)
print("Len of triplets:", len(filtered_tst_dataset[1]["topk_linearized_triplets"]))
tst_dataloader = DataLoader(filtered_tst_dataset, batch_size=1, shuffle=False)


In [ ]:
path = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv4-n1000_e50_k30_cosine/checkpoint_epoch_50"
joint_trainer_ablation = JointTrainer.load_checkpoint(path)
evaluate_answer_and_path_coverage(tst_dataloader,
    joint_trainer_ablation,
    top_k=30)

In [ ]:
path = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv4-n1000-e50-k50_cosine/checkpoint_epoch_50"
joint_trainer_ablation = JointTrainer.load_checkpoint(path)
evaluate_answer_and_path_coverage(tst_dataloader,
    joint_trainer_ablation,
    top_k=50)

In [ ]:
path = "/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv4-n1000_e50_k150_cosine/checkpoint_epoch_50"
joint_trainer_ablation = JointTrainer.load_checkpoint(path)
evaluate_answer_and_path_coverage(tst_dataloader,
    joint_trainer_ablation,
    top_k=150)

In [ ]:
joint_trainer_best = JointTrainer.load_checkpoint("/mnt/LS226/LS25/sourav23099/cwq/cwq-rml-v2/model/architecture-v8/v7-rv8-n1000-e30_cosine/checkpoint_best_epoch_30")
evaluate_answer_and_path_coverage(tst_dataloader,
    joint_trainer_best,
    top_k=100)